# 从零读懂：S&P 500 期权价格 Z-score 仪表盘
## 原代码保留 · 中文逐步解释 · 可运行的小练习

**对应文件：** `BQNT_SP500_Options_ZScore(1).ipynb`  
**阅读对象：** 刚开始学习 Python、pandas 和 Notebook 的读者。

这份文件不是重新设计一套策略，而是陪你读懂原文件：**每一段输入什么、做了什么、输出什么，以及为什么这样写。**

> **先记住一句话：** 原程序给当前指数成分股挑选当前的一对 Call / Put 期权，然后把每张选中合约的价格，与**这张合约自己**过去的价格比较，最后显示异常程度。

### 本教学版与原文件的关系

原文件共有 **12 个单元格，其中 5 个是代码单元格**。本教学版沿用原来的顺序：设置 → 检查与计算 → Bloomberg 取数与选合约 → 交互界面 → 启动 → 结果解释。

计算、报价检查、合约选择和界面函数均保留原代码，只按完整函数拆成较小的可执行单元格；没有暗中替换算法、字段、默认窗口或数据源。原来的英文说明也保留在可展开区域中。

**唯一的启动方式调整：** 原最后 6 行启动代码放在一个默认关闭的 `RUN_BLOOMBERG` 开关下。这样，没有 Bloomberg 的读者也能运行教学练习。在有权限的 BQNT 中，把该开关改为 `True`，即可执行原来的启动语句；真正取数仍需点击原界面的按钮。

**新增练习的边界：** 标为「教学练习」的数据全部是人为写的小数字，变量名以 `lesson_` 开头。它们只演示函数，不会成为仪表盘的备用行情，更不是实际市场结果。

**验证边界：** 本文件的离线练习、Notebook 结构和代码语法已检查；未连接 Bloomberg，未验证真实 BQL 请求、历史报价权限或 BQNT 前端的实时交互效果。原文件关于其先前测试的描述是原作者的说明，不等于本次已验证 Bloomberg。

<a id="contents"></a>
## 阅读导航

| 章节 | 读完后你能回答的问题 |
|---|---|
| [0. 先看全貌](#overview) | 这是在分析什么？哪些事情它没有做？ |
| [1. Settings：工具与参数](#settings) | `import`、变量、默认值是什么意思？ |
| [2. Validation：数据检查与 Z-score](#calculation) | 表格怎样变形？365 天窗口怎样计算？ |
| [3. BQL：请求、选合约、过滤报价](#retrieval) | 数据怎样一步步从 Bloomberg 进入计算函数？ |
| [4. Widgets：表格、图形与按钮](#widgets) | 为什么点击按钮才加载？更改控件会发生什么？ |
| [5. Launch：如何运行](#launch) | 没有 Bloomberg 时怎样学？有权限时怎样启动？ |
| [6. Interpretation：读懂结果与空值](#interpretation) | Z-score 大就值得交易吗？空白是什么原因？ |
| [7. 自测与速查](#review) | 能否自己说清整条数据链？ |
| [8. 来源与验证记录](#sources) | 哪些解释来自原文件？哪些是教学补充？ |

### Notebook 怎么读、怎么运行？〔教学补充〕

**Markdown 单元格**是说明文字，不会获取行情。解释中的灰色代码片段只是引用，不要把半个函数单独复制运行。**Code 单元格**才是真正可以执行的 Python。

通常选中一个代码单元格后按 `Shift + Enter`，执行它并进入下一个单元格。运行顺序很重要：先运行导入和函数定义，再运行调用它们的练习。内核会记住已经创建的变量与函数；重启内核会清空这些记忆。

第一遍可以只读文字、观察已经保存的小练习输出。第二遍从上到下执行全部单元格，保持 `RUN_BLOOMBERG = False`。第三遍再在有权限的 BQNT 中打开该开关。

**没有输出不等于没运行成功。** 执行 `def ...` 通常只是登记一份函数说明书，不会立即做里面的工作。要等到 `函数名(...)` 被调用时，函数体才开始执行。原文件里大部分代码都属于“先定义，后调用”。

<a id="overview"></a>
# 0. 先看全貌：这份程序究竟在干什么？

**依据：原单元格 1、3、7、9、12。**

把它想成一个“选合约 → 拿历史 → 比较今天 → 展示结果”的流水线：

```text
当前 SPX Index 成分股名单
    ↓
这些股票约两年的日度股价 → 确定共同展示日期 asof
    ↓
逐只股票查询当前期权链
    ↓
选择同到期日、同行权价的一张 Call 和一张 Put
    ↓
请求这两张固定合约自己的历史 Bid / Ask
    ↓
过滤不合格报价，计算中间价 Mid
    ↓
整理成 date / ticker / spot / call_mid / put_mid 五列表
    ↓
对每只股票、每个指标计算历史均值、标准差和 Z-score
    ↓
按 |Z| 排序的表格 + 单只股票的历史图
```

### 它分析的是成分股期权，不是直接分析 SPX 指数期权

`INDEX = "SPX Index"` 在这里用于获取指数成员。之后 `bq.univ.options(ticker)` 对**每个成员的股票代码**查询期权链。不要看到文件名里有 S&P 500，就以为选出的都是 SPX 指数本身的期权。

### 它不是交易机器人，也不是回测报告

原代码没有下单、买卖规则、仓位、交易成本、收益率计算或策略回测。它没有计算隐含波动率，也没有使用定价模型补出缺失价格。它使用的是当前成分股名单，不是逐个历史时点重建的指数成员。

### 最容易误解的地方：30 天只是当前的选合约目标

假设当前选中一张约 30 天后到期的期权，程序会固定这张合约的 ID，往回查询**同一张合约**。在更早的时候，它剩余的到期时间更长，股价与行权价的距离也可能不同。

程序**没有**在每个历史日期重新选择“当时距到期 30 天、当时最接近平值”的另一张期权。所以图中不是统一剩余期限、统一平值程度的连续序列。

**由代码直接推得：** 刷新时可能挑出另一张当前合约，此后整条历史线也就换成新合约的历史。不同刷新结果之间，不能只看股票代码相同就认定是同一条期权序列。

<details>
<summary>展开查看原文件第 1 个单元格的英文说明（原文保留）</summary>

# S&P 500 option-price Z-scores — Bloomberg BQNT only

Run all cells in an entitled Bloomberg BQuant notebook, then click **Load / refresh Bloomberg**. All market-data requests use `bql.Service()`. There is no synthetic-data mode, CSV input, or unconfigured premium-field callback.

**Methodology:** for every current S&P 500 constituent, select a current listed call/put pair near the chosen target expiry and closest strike to spot. Display each selected contract's latest available daily bid/ask midpoint and compare that exact contract with its own previous 365 calendar days of prices.

**This is not a constant-maturity / historically rolled ATM series.** Unlike the standardized-series proposal in the previous starter, this implementation uses currently selected listed contracts throughout their own histories. Maturity and moneyness change through that history. Recently listed contracts may have no full-year Z-score; missing scores remain blank.

The score excludes the current observation, requires at least 200 valid prior observations and 365 days of elapsed history, and uses sample standard deviation. About two years of daily history are requested to support rolling charts. The current universe is not a point-in-time index backtest.

**Validation:** the arithmetic, quote filters, contract selection and response joins were tested locally using independent test fixtures and a mocked transport. The proprietary Bloomberg requests and notebook widget rendering have not been tested in a live BQNT session. Field availability, historical bid/ask coverage, fill semantics and licensing must be verified there. This is a prototype, not a production-validated pricing monitor.


</details>

## 0.1 先认识变量里的几个词〔教学术语补充〕

| 名词 | 在这份文件中的含义 | 不要误读为 |
|---|---|---|
| `ticker` / `ID` | 证券标识；具体属于股票还是期权，要看所在步骤 | 一个字段在所有表里都只代表股票 |
| `spot` | 标的股票价格；取数使用 `px_last` 的日度历史 | 期权价格 |
| Call / Put | 看涨期权 / 看跌期权 | 两种股票 |
| `Expiry` / `DTE` | 到期日期 / 从 `asof` 到到期日的日历天数 | 交易日数量 |
| `Strike` | 行权价；程序用它与 `spot` 的距离找当前近似平值合约 | 当前期权报价 |
| Bid / Ask / Mid | 买价 / 卖价 / 二者的算术平均值 | 成交价或保证可成交的价格 |
| Premium | 这里讨论的期权价格 | 收益率或隐含波动率 |
| `asof` | 本次统一展示和计算所使用的市场日期 | 每张期权都在同一秒更新 |
| `NaN` / `NaT` / `None` | 分别常用于缺失数值、缺失日期、尚未给值 | 数字 0 |
| Z-score | 当前值离自己的历史均值有多少个历史标准差 | 自动的买卖信号 |

原文件要求价格口径是 **USD per underlying share（每一标的股对应的美元价格）**。代码没有将价格乘以某个合约乘数，也没有自动核实所有调整后合约的交割内容。这里只能解释要求，不能凭字段名证明真实返回数据满足它。

<a id="settings"></a>
# 1. Settings：先准备工具，再设定参数

**对应原单元格 2、3。** 下一格保留原第 3 个代码单元格。执行它只是导入工具、创建常量；不会向 Bloomberg 请求数据。

<details>
<summary>展开查看原文件第 2 个单元格的英文说明（原文保留）</summary>

## 1. Settings

`MAX_STOCKS = None` requests every current constituent. Set it to `5` for an initial connection check. Target DTE and the allowed expiry band can also be changed in the dashboard before refreshing.

The only data path is Bloomberg BQL. `MODE = "BQNT"` is an informational label, not a Bloomberg API parameter. The BQL session is instantiated when the interface is launched.


</details>

In [1]:
"""Bloomberg BQNT-only S&P 500 listed-option price Z-score dashboard.

No synthetic-data mode, CSV loader, or unconfigured premium-field callbacks.
Run in an entitled Bloomberg BQuant notebook. Live BQL execution is untested here.

Method: select a CURRENT listed call/put pair near the target expiry and ATM
strike; compare each exact contract's premium to its OWN prior 365-day history.
This is NOT a historical constant-maturity / constant-moneyness option series.
"""
from __future__ import annotations

from functools import reduce
from html import escape
from typing import Callable, Optional

import numpy as np
import pandas as pd

MODE = "BQNT"  # Informational only: Bloomberg is the only data path.
INDEX = "SPX Index"
TARGET_DTE = 30
EXPIRY_TOLERANCE_DAYS = 14
MIN_DTE = 7
WINDOW_DAYS = 365
MIN_OBSERVATIONS = 200
BATCH_SIZE = 40
MAX_STOCKS = None  # None = every current constituent; 5 = connection test.

## 1.1 从上到下读导入语句

**依据：原单元格 3，第 1–17 行；下列类比是教学补充。**

开头的 `""" ... """` 是整段说明文字。`#` 开头的是单行注释。这些文字用于帮助读者，不是行情请求。

`from __future__ import annotations` 与函数的类型标注有关。原文件标注的 Python 版本为 3.10；在这种写法下，类型标注不会按普通表达式立即求值。初学时先记住：它不会改变 Z-score 公式，也不会连接 Bloomberg。

| 语句 | 准备什么工具 | 稍后在哪儿出现 |
|---|---|---|
| `from functools import reduce` | 反复把两个对象合成一个 | 把多个 BQL 字段结果逐步合成一张表 |
| `from html import escape` | 把特殊字符转成安全显示的 HTML 文本 | 将错误信息、搜索相关展示文本放进界面 |
| `from typing import Callable, Optional` | 描述参数预期类型 | `progress` 回调、允许为空的参数 |
| `import numpy as np` | 数值工具，使用短名字 `np` | 检查无穷值、使用 `np.nan` |
| `import pandas as pd` | 表格与日期工具，使用短名字 `pd` | DataFrame、日期转换、透视和滚动计算 |

`as np` / `as pd` 只是起别名，不是复制数据。`raw: pd.DataFrame` 表示“希望 raw 是一张 pandas 表”；它不等于自动验证。原作者又写了显式的 `isinstance` 等检查，才真正拦截不合格输入。

**执行到这里的结果：** 内核认识 `np`、`pd` 和这些配置变量了，但还没有 `history` 行情表。

## 1.2 参数逐个解释：哪些影响选合约，哪些影响统计？

**依据：原单元格 3，第 19–27 行及其后续使用位置。**

| 变量 | 原默认值 | 具体用途 |
|---|---|---|
| `MODE` | `"BQNT"` | 说明性标签；代码没有据此切换数据模式 |
| `INDEX` | `"SPX Index"` | 获取当前成分股名单 |
| `TARGET_DTE` | `30` | 当前选合约时希望剩余约 30 个日历日 |
| `EXPIRY_TOLERANCE_DAYS` | `14` | 到期天数允许偏离目标最多 14 天 |
| `MIN_DTE` | `7` | 剩余到期时间不得小于 7 天 |
| `WINDOW_DAYS` | `365` | 每个计算日往前看的日历时间长度 |
| `MIN_OBSERVATIONS` | `200` | 该窗口内至少需要 200 个非缺失的历史观察值 |
| `BATCH_SIZE` | `40` | 股价和期权报价历史每批请求的证券数量；不是并发线程数 |
| `MAX_STOCKS` | `None` | 不额外截断返回的成分股名单；设为 5 时取排序后的前 5 个 |

默认选合约区间满足 `abs(DTE - 30) <= 14` 且 `DTE >= 7`，因此是 **16–44 天（含边界）**。这不保证存在符合条件、能配成对的合约。

`None` 不是 0。这里它表示“不设这个数量上限”。大写变量名是作者表示配置常量的习惯，Python 本身仍允许重新赋值。

**初次连接沿用原说明的做法：** 将上方 `MAX_STOCKS` 改为 `5`，重新运行设置格，再继续执行后续定义与启动格。它只缩小股票数量，不缩短 365 天的统计要求。

In [2]:
# 教学练习 1：只观察配置，不请求行情。
from IPython.display import display  # 为后面的教学表格准备显示函数
print("数据路径标签:", MODE)
print("目标 DTE:", TARGET_DTE)
print("默认允许的 DTE 区间:", max(MIN_DTE, TARGET_DTE - EXPIRY_TOLERANCE_DAYS),
      "至", TARGET_DTE + EXPIRY_TOLERANCE_DAYS)
print("观察窗（日历天）:", WINDOW_DAYS, "；最少历史观察值:", MIN_OBSERVATIONS)

数据路径标签: BQNT
目标 DTE: 30
默认允许的 DTE 区间: 16 至 44
观察窗（日历天）: 365 ；最少历史观察值: 200


<a id="calculation"></a>
# 2. Validation and rolling calculations：先检查，再计算

**对应原单元格 4、5。** 这一部分只处理已经拿到的表格，不直接连接 Bloomberg。

这里有两个主要函数：`validate_history(raw)` 负责“这张表有没有明显格式或价格错误”；`compute_dashboard(raw, ...)` 负责“怎样从合格表格算出结果”。

<details>
<summary>展开查看原文件第 4 个单元格的英文说明（原文保留）</summary>

## 2. Validation and rolling calculations

</details>

## 2.1 输入表长什么样？

**依据：原单元格 5，第 1–2 行。** `REQUIRED` 是必须有的列名清单；`METRICS` 将界面名称映射为程序内部使用的短键名。

| 列名 | 一行中的意思 | 缺失是否允许 |
|---|---|---|
| `date` | 一个市场日期，不带时区 | 不允许 |
| `ticker` | 这行对应的股票标识 | 不允许为空或空白字符串 |
| `spot` | 该日股价 | 允许缺失；有值时须大于 0 |
| `call_mid` | 当前选定 Call 在该日的中间价 | 允许缺失；有值时不得小于 0 |
| `put_mid` | 当前选定 Put 在该日的中间价 | 允许缺失；有值时不得小于 0 |

同一 `date + ticker` 只能有一行。表中可能跨越很多天、很多只股票，这种结构叫“长表”。它还没有把每只股票变成一列。

**重要边界：** 这五列没有期权合约 ID，因此仅靠这张表无法证明整段历史都属于同一张合约。原程序通过取数阶段选择、固定和记录合约 ID 来组织数据；检查函数本身不会从五列重新证明这一点。

In [3]:
REQUIRED = ["date", "ticker", "spot", "call_mid", "put_mid"]
METRICS = {"Call Z": "call", "Put Z": "put", "Spread Z": "spread"}

## 2.2 `validate_history`：数据的入场检查

**输入：** `raw`，一张 pandas DataFrame。  
**输出：** `d`，只保留必要列、整理日期与类型、排序后的新表。  
**失败时：** 抛出带说明文字的 `ValueError`，而不是默默修成另一个数据集。

下面先分段理解，随后一次性运行完整函数。

### 2.2.1 检查是不是非空表，以及必要列是否齐全

**原文件定位：第 5 个单元格，第 5–17 行。以下是阅读片段，不要单独运行。**

```python
def validate_history(raw: pd.DataFrame) -> pd.DataFrame:
    """Validate daily observations for each currently selected listed-option pair.

    Prices must be USD per underlying share; dates must be timezone-naive US
    market-date labels. Missing observations are allowed but never filled.
    The caller must verify fixed contract identifiers and price conventions;
    the five-column schema alone cannot establish these properties.
    """
    if not isinstance(raw, pd.DataFrame) or raw.empty:
        raise ValueError("The history loader must return a nonempty DataFrame.")
    missing = set(REQUIRED).difference(raw.columns)
    if missing:
        raise ValueError("Missing input columns: " + ", ".join(sorted(missing)))
```

`def` 开始定义函数；缩进表示哪些语句属于函数内部。`-> pd.DataFrame` 说明预期返回一张表，不是“把表导出到文件”。

`not isinstance(raw, pd.DataFrame) or raw.empty` 表示：不是 DataFrame，**或者**是一张空表，就报错。`raise ValueError(...)` 主动中止当前调用，附带错误原因。

`set(REQUIRED).difference(raw.columns)` 求“要求有、实际却没有”的列。比如缺少 `put_mid`，`missing` 中就会有它。`sorted` 排序后，`", ".join(...)` 将列名拼成便于阅读的报错信息。

### 2.2.2 复制数据、规范日期与代码、拒绝重复行

**原文件定位：第 5 个单元格，第 18–31 行。以下是阅读片段，不要单独运行。**

```python
    d = raw[REQUIRED].copy()
    d["date"] = pd.to_datetime(d["date"], errors="raise")
    if d["date"].isna().any():
        raise ValueError("Dates cannot be missing.")
    if d["date"].dt.tz is not None:
        raise ValueError("Use timezone-naive US market-date labels, not timestamps.")
    d["date"] = d["date"].dt.normalize()
    if d["ticker"].isna().any():
        raise ValueError("Ticker values cannot be missing.")
    d["ticker"] = d["ticker"].astype(str).str.strip()
    if d["ticker"].eq("").any():
        raise ValueError("Ticker values cannot be blank.")
    if d.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate date/ticker rows: select a single consistent option series first.")
```

`raw[REQUIRED]` 选出要求的五列，额外列不会进入计算；`.copy()` 创建独立副本，避免直接改动调用者原表。

`pd.to_datetime(..., errors="raise")` 把日期转成可比较的日期类型，无法转换就报错。`.isna().any()` 表示“有没有至少一个缺失值”。带时区日期会被拒绝；`.dt.normalize()` 将时间部分归到该日零点，**不是时区转换**。

`astype(str).str.strip()` 把股票标识转成字符串并去掉两端空白。`eq("")` 检查空字符串。这里不验证某个字符串是不是 Bloomberg 真实证券代码。

`duplicated(["date", "ticker"])` 检查日期和股票的组合是否重复。由于前面已把时间归到零点，同一股票同一天不同时间的两行也会被视为重复，而不是自动取平均。

### 2.2.3 检查数字，排序后返回

**原文件定位：第 5 个单元格，第 32–39 行。以下是阅读片段，不要单独运行。**

```python
    for col in ["spot", "call_mid", "put_mid"]:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(float)
        if np.isinf(d[col]).any():
            raise ValueError(f"Infinite values in {col}.")
        invalid = d[col].le(0) if col == "spot" else d[col].lt(0)
        if invalid.any():
            raise ValueError(f"Invalid negative/zero price in {col}.")
    return d.sort_values(["date", "ticker"]).reset_index(drop=True)
```

`for col in [...]` 表示对股价、Call 中间价、Put 中间价依次做同样的工作。`to_numeric(..., errors="raise").astype(float)` 将可转换值转成浮点数；无法转换时不会悄悄当成 0。

`np.isinf` 检查正负无穷值。`le(0)` 是“小于等于 0”，`lt(0)` 是“小于 0”。条件表达式让股价采用更严格的 `<= 0` 检查，期权中间价采用 `< 0` 检查。

**这里允许期权值为 0；后面的真实 Bid / Ask 过滤则要求 Ask > 0。两个函数的检查层次不同。** 缺失价格仍可保留，不会被填补。

最后按日期、股票排序；`reset_index(drop=True)` 将旧行号丢弃，换成整齐的 0、1、2……；`return` 把清理后的表交给调用者。

### 运行完整函数：`validate_history`

下一格是原单元格 5 第 5–39 行原代码。运行后只定义函数，不会立刻检查或查询行情。

In [4]:
def validate_history(raw: pd.DataFrame) -> pd.DataFrame:
    """Validate daily observations for each currently selected listed-option pair.

    Prices must be USD per underlying share; dates must be timezone-naive US
    market-date labels. Missing observations are allowed but never filled.
    The caller must verify fixed contract identifiers and price conventions;
    the five-column schema alone cannot establish these properties.
    """
    if not isinstance(raw, pd.DataFrame) or raw.empty:
        raise ValueError("The history loader must return a nonempty DataFrame.")
    missing = set(REQUIRED).difference(raw.columns)
    if missing:
        raise ValueError("Missing input columns: " + ", ".join(sorted(missing)))
    d = raw[REQUIRED].copy()
    d["date"] = pd.to_datetime(d["date"], errors="raise")
    if d["date"].isna().any():
        raise ValueError("Dates cannot be missing.")
    if d["date"].dt.tz is not None:
        raise ValueError("Use timezone-naive US market-date labels, not timestamps.")
    d["date"] = d["date"].dt.normalize()
    if d["ticker"].isna().any():
        raise ValueError("Ticker values cannot be missing.")
    d["ticker"] = d["ticker"].astype(str).str.strip()
    if d["ticker"].eq("").any():
        raise ValueError("Ticker values cannot be blank.")
    if d.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate date/ticker rows: select a single consistent option series first.")
    for col in ["spot", "call_mid", "put_mid"]:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(float)
        if np.isinf(d[col]).any():
            raise ValueError(f"Infinite values in {col}.")
        invalid = d[col].le(0) if col == "spot" else d[col].lt(0)
        if invalid.any():
            raise ValueError(f"Invalid negative/zero price in {col}.")
    return d.sort_values(["date", "ticker"]).reset_index(drop=True)

In [5]:
# 教学练习 2：完全虚构的小表，观察清理前后。
lesson_raw = pd.DataFrame({
    "date": ["2024-01-02", "2024-01-01", "2024-01-02", "2024-01-01"],
    "ticker": [" DEMO_A ", "DEMO_A", "DEMO_B", "DEMO_B"],
    "spot": [101, 100, 51, 50],
    "call_mid": [3.0, 2.0, np.nan, 1.5],
    "put_mid": [1.0, 1.2, 0.8, 0.9],
    "extra_note": ["教学", "教学", "教学", "教学"],
})
lesson_clean = validate_history(lesson_raw)
print("清理后：额外列被排除，代码去掉两端空白，并按日期、股票排序。")
display(lesson_clean)
assert list(lesson_clean.columns) == REQUIRED
assert lesson_clean["call_mid"].isna().sum() == 1
assert lesson_raw.loc[0, "ticker"] == " DEMO_A "  # 原输入没有被就地修改

清理后：额外列被排除，代码去掉两端空白，并按日期、股票排序。


,date,ticker,spot,call_mid,put_mid
0,2024-01-01,DEMO_A,100.0,2.0,1.2
1,2024-01-01,DEMO_B,50.0,1.5,0.9
2,2024-01-02,DEMO_A,101.0,3.0,1.0
3,2024-01-02,DEMO_B,51.0,NaN,0.8


In [6]:
# 教学练习 3：故意制造重复行；捕捉错误，让整个教学文件仍能继续运行。
lesson_duplicate = pd.concat([lesson_clean, lesson_clean.iloc[[0]]], ignore_index=True)
try:
    validate_history(lesson_duplicate)
except ValueError as lesson_error:
    print("这次报错是预期结果：", lesson_error)
else:
    raise AssertionError("重复行本应被拒绝。")

这次报错是预期结果： Duplicate date/ticker rows: select a single consistent option series first.


## 2.3 `compute_dashboard`：先理解公式，再看实现

**依据：原单元格 5 第 42–116 行。** 输入是五列长表；返回一个字典，里面包含最新表格、历史得分、均值、标准差、观察值数量和状态文字。

对于某只股票某个指标在日期 `t` 的值，原算法写成数学形式是：

$$
Z_t = \frac{x_t-\mu_t}{s_t},\qquad
\mu_t=\frac{1}{n}\sum_{i\in W_t} x_i,\qquad
s_t=\sqrt{\frac{\sum_{i\in W_t}(x_i-\mu_t)^2}{n-1}}
$$

其中 `W_t` 是 `[t - 365 天, t)` 内该指标的非缺失观察值。左端点包含，右端点不包含，所以**今天的值不参与今天的历史均值和标准差**。这是每个合约/指标沿时间轴的比较，不是把所有股票放在一起做横截面标准化。

### 先手算一个极小例子〔教学补充〕

假设历史值为 `8、10、12`，今天为 `14`：历史均值是 `10`；平方偏差之和是 `4 + 0 + 4 = 8`；样本方差是 `8 / (3 - 1) = 4`；样本标准差是 `2`，因此 `Z = (14 - 10) / 2 = 2`。

这只说明“比自己的这段历史均值高 2 个样本标准差”。它没有证明合理价格是多少，也没有保证未来会回到均值。这里三个数字的练习不是原程序的 200 次观察值要求。

In [7]:
# 教学练习 4：手算公式对应的 Python；不调用 Bloomberg。
lesson_prior = pd.Series([8.0, 10.0, 12.0])
lesson_today = 14.0
lesson_mean = lesson_prior.mean()
lesson_std = lesson_prior.std(ddof=1)
lesson_z = (lesson_today - lesson_mean) / lesson_std
print("历史均值 =", lesson_mean, "；样本标准差 =", lesson_std, "；Z =", lesson_z)
assert lesson_mean == 10.0 and lesson_std == 2.0 and lesson_z == 2.0

历史均值 = 10.0 ；样本标准差 = 2.0 ；Z = 2.0


### 2.3.1 参数、日期和最早的防错检查

**原文件定位：第 5 个单元格，第 42–67 行。以下是阅读片段，不要单独运行。**

```python
def compute_dashboard(
    raw: pd.DataFrame,
    basis: str = "Raw premium",
    window_days: int = 365,
    min_obs: int = 200,
    asof: Optional[str] = None,
) -> dict:
    """Calculate per-stock trailing z-scores, never cross-sectional z-scores.

    Baseline at t: dates in [t-window_days, t); today's value is excluded.
    A score requires min_obs nonmissing prior observations AND at least
    window_days of elapsed history for that stock/metric. Standard deviation
    uses ddof=1. A zero or invalid standard deviation produces NaN, not zero.
    This is a 365-day convention, not an exact anniversary in leap years.
    """
    if window_days < 2 or min_obs < 2:
        raise ValueError("window_days and min_obs must both be >= 2.")
    if basis not in {"Raw premium", "Premium / spot"}:
        raise ValueError("Unknown price basis.")
    d = validate_history(raw)
    stamp = pd.Timestamp(asof).normalize() if asof else d["date"].max()
    if stamp.tzinfo is not None:
        raise ValueError("asof must be a timezone-naive market date.")
    d = d.loc[d["date"].le(stamp)].copy()
    if d.empty:
        raise ValueError("There are no observations on or before the requested date.")
```

`basis` 决定用原始期权价格，还是期权价格相对股价的比例。`window_days` 与 `min_obs` 默认分别为 365 和 200；`asof=None` 表示没有另外指定展示日期。

代码拒绝过小的窗口和观察值要求，也拒绝未定义的价格口径。然后再次调用 `validate_history`，保证独立使用计算函数时同样有入口检查。

`stamp = ... if asof else ...`：有传入日期就用它，否则用数据中最大的日期。随后只保留 `date <= stamp` 的行。**这里的 `asof` 只是计算截止日期，不会让取数模块自动重建那个历史日期的成分股与期权链。**

类型标注 `Optional[str]` 表达“字符串或 None”的意图，但它本身不做运行时强制转换检查；原界面实际传入的是可被 `pd.Timestamp` 接受的日期对象。

### 2.3.2 长表转宽表：日期做行，股票做列

**原文件定位：第 5 个单元格，第 68–75 行。以下是阅读片段，不要单独运行。**

```python
    tickers = sorted(d["ticker"].unique())
    dates = pd.DatetimeIndex(sorted(set(d["date"]) | {stamp}), name="date")
    price = {
        col: d.pivot(index="date", columns="ticker", values=col).reindex(
            index=dates, columns=tickers
        )
        for col in ["spot", "call_mid", "put_mid"]
    }
```

`unique()` 获取不重复的股票标识，`sorted(...)` 固定显示顺序。`set(d["date"]) | {stamp}` 是日期集合的并集：即使截止日期原本没有数据，也把它加入索引，后面取 `.loc[stamp]` 就有位置可取。

`pivot(index="date", columns="ticker", values=col)` 将长表变成宽表。一张表只放一个字段，例如 `call_mid`：行是日期，列是股票，格子里是那天那只股票对应 Call 的中间价。

`.reindex(index=dates, columns=tickers)` 将三张价格表的行和列对齐。它不生成交易日历，也不自动补上所有日历日期；它使用的是数据中出现的日期加上 `stamp`。新出现但没有数据的位置是缺失值，不是昨日价格。

外层的字典推导式，为 `spot`、`call_mid`、`put_mid` 各做一次，得到 `price` 字典。

In [8]:
# 教学练习 5：用前面的虚构表观察 pivot。
lesson_wide = lesson_clean.pivot(index="date", columns="ticker", values="call_mid")
print("一列数值的长表 → 日期 × 股票的宽表：")
display(lesson_wide)
assert lesson_wide.shape == (2, 2)

一列数值的长表 → 日期 × 股票的宽表：


ticker,DEMO_A,DEMO_B
date,,
2024-01-01,2.0,1.5
2024-01-02,3.0,NaN


### 2.3.3 两种价格口径，以及 Spread 的正确含义

**原文件定位：第 5 个单元格，第 76–81 行。以下是阅读片段，不要单独运行。**

```python
    call, put = price["call_mid"], price["put_mid"]
    if basis == "Premium / spot":
        call = 100.0 * call.div(price["spot"])
        put = 100.0 * put.div(price["spot"])
    values = {"call": call, "put": put, "spread": call - put}
    scores, counts, means, stds, statuses = {}, {}, {}, {}, {}
```

`call, put = ...` 同时取得两张期权价格表。

在 `Raw premium` 下，`call`、`put` 使用原始中间价。在 `Premium / spot` 下，程序实际计算 `100 × 期权价格 / 当日股价`；所以股价 100、期权价格 2 会得到数值 2，含义是股价的 **2%**，不是比例小数 0.02。

这次换算不会覆盖 `price` 里保存的原始价格。股价缺失时，相应的归一化指标也会缺失。

`values` 保存三个待计算指标：Call、Put，以及 `call - put`。**`Spread Z` 是差值序列本身的 Z-score，不是 `Call Z - Put Z`，也不是 Ask 减 Bid 的买卖价差。**

末尾一次创建五个空字典，后续分别装入三类指标的得分、数量、均值、标准差和状态。

### 2.3.4 时间窗口、均值、标准差和观察值数量

**原文件定位：第 5 个单元格，第 82–86 行。以下是阅读片段，不要单独运行。**

```python
    delta = pd.Timedelta(days=window_days)
    for metric, x in values.items():
        rolling = x.rolling(f"{window_days}D", closed="left", min_periods=2)
        mean, std = rolling.mean(), rolling.std(ddof=1)
        count = x.rolling(f"{window_days}D", closed="left", min_periods=0).count()
```

`pd.Timedelta(days=window_days)` 表示一个日历时间长度。`for metric, x in values.items()` 依次处理 Call、Put、Spread，每次 `x` 都是一张“日期 × 股票”的表。

`rolling(f"{window_days}D", closed="left", min_periods=2)`：窗口按日期而不是固定行数移动；`closed="left"` 对应 `[t-window_days, t)`，排除当日；`min_periods=2` 是先生成滚动统计量所需的最少有效值，不等于最后允许展示得分的 200 次要求。

`.std(ddof=1)` 使用样本标准差，分母为 `n-1`。另一个滚动对象用 `.count()` 统计窗口内非缺失值的数量；这里设 `min_periods=0`，是为了历史不足时仍能得到数量信息。

**不是 365 行，不是 365 个交易日，也不是所有历史从开头一路累积。** 窗口只保留日期落在指定区间内的观察值。365 天在闰年也仍是固定 365 天，不是自动按周年日切分。

教学补充出处：[pandas rolling 文档（S1）](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html)；[样本标准差参数文档（S2）](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Rolling.std.html)。这里说明的是参数含义，不代表 Bloomberg 环境安装的是文档展示的版本。

### 2.3.5 两道历史门槛、标准差检查和状态优先级

**原文件定位：第 5 个单元格，第 87–101 行。以下是阅读片段，不要单独运行。**

```python
        mature = pd.DataFrame(False, index=dates, columns=tickers)
        for ticker in tickers:
            first = x[ticker].first_valid_index()
            if first is not None:
                mature[ticker] = dates >= first + delta
        eligible = count.ge(min_obs) & mature
        z = ((x - mean) / std.where(std.gt(1e-12))).where(eligible)
        scores[metric], counts[metric] = z, count
        means[metric], stds[metric] = mean, std
        status = pd.Series("OK", index=tickers, dtype=object)
        status.loc[~std.loc[stamp].gt(1e-12)] = "Zero/invalid standard deviation"
        status.loc[count.loc[stamp].lt(min_obs)] = f"Fewer than {min_obs} prior observations"
        status.loc[~mature.loc[stamp]] = f"Less than {window_days} days of history"
        status.loc[x.loc[stamp].isna()] = "No value at the as-of date"
        statuses[metric] = status
```

`mature` 最初是一张全为 False 的表。每只股票、每个指标各自找 `first_valid_index()`，即**这次输入历史里第一个非缺失值的日期**，不是从交易所查询来的上市日期。

`dates >= first + delta` 判断是否已经经过至少一个完整窗口长度。`eligible = count.ge(min_obs) & mature` 要求“窗口内有效观察值够多”与“已跨过足够长时间”同时成立。`&` 是逐格的逻辑“并且”。

`std.where(std.gt(1e-12))` 只保留大于 `0.000000000001` 的标准差；否则变成缺失值，避免除以 0 或极小分母。最后的 `.where(eligible)` 再把历史条件不合格的位置留空。

**一张新合约即使有足够多的输入行，若首个有效值距当前不足 365 天，原默认规则仍不给有效 Z。** 反过来，历史跨度很长但窗口内只有少量非缺失值，也不合格。

状态从 `OK` 开始，然后按代码顺序覆盖。最终优先显示：**当前无值 → 历史跨度不足 → 观察值不足 → 标准差无效**。一条状态不代表只存在这一个问题。三个指标各算各的，所以同一股票可能 Call 有 Z、Put 无 Z、Spread 也无 Z。

### 2.3.6 从整段历史中抽出当前行，再打包返回

**原文件定位：第 5 个单元格，第 103–116 行。以下是阅读片段，不要单独运行。**

```python
    latest = pd.DataFrame(index=pd.Index(tickers, name="Ticker"))
    latest["Spot"] = price["spot"].loc[stamp]
    latest["Call mid"] = price["call_mid"].loc[stamp]
    latest["Put mid"] = price["put_mid"].loc[stamp]
    for label, metric in METRICS.items():
        latest[label] = scores[metric].loc[stamp]
    quote_rows = d.loc[d[["call_mid", "put_mid"]].notna().any(axis=1)]
    last_quote = quote_rows.groupby("ticker")["date"].max()
    latest["Last option date"] = last_quote.reindex(tickers)
    return dict(
        latest=latest, scores=scores, values=values, price=price,
        counts=counts, means=means, stds=stds, statuses=statuses,
        asof=stamp, window_days=window_days, min_obs=min_obs, basis=basis,
    )
```

`latest` 用股票作为索引，每只股票一行。前三列始终取自原始 `price`；即使选择比例口径，表格的 `Call mid` 与 `Put mid` 也仍是原始中间价。

循环将三个指标在 `stamp` 的得分取出来。`quote_rows` 保留至少有一个 Call 或 Put 中间价的日期；`groupby("ticker")["date"].max()` 得到该股票两边任一有效报价出现过的最近日期。

因此 `Last option date` 不是“Call 与 Put 同时有效的最后日期”。界面实际另行显示的 `Call last date` 和 `Put last date` 来自取数模块的合约元数据。

`return dict(...)` 一次返回多个结果，不会自动把文件保存到硬盘。

### 运行完整函数：`compute_dashboard`

下一格保留原单元格 5 第 42–116 行。执行它后，才可以运行随后的教学计算。

In [9]:
def compute_dashboard(
    raw: pd.DataFrame,
    basis: str = "Raw premium",
    window_days: int = 365,
    min_obs: int = 200,
    asof: Optional[str] = None,
) -> dict:
    """Calculate per-stock trailing z-scores, never cross-sectional z-scores.

    Baseline at t: dates in [t-window_days, t); today's value is excluded.
    A score requires min_obs nonmissing prior observations AND at least
    window_days of elapsed history for that stock/metric. Standard deviation
    uses ddof=1. A zero or invalid standard deviation produces NaN, not zero.
    This is a 365-day convention, not an exact anniversary in leap years.
    """
    if window_days < 2 or min_obs < 2:
        raise ValueError("window_days and min_obs must both be >= 2.")
    if basis not in {"Raw premium", "Premium / spot"}:
        raise ValueError("Unknown price basis.")
    d = validate_history(raw)
    stamp = pd.Timestamp(asof).normalize() if asof else d["date"].max()
    if stamp.tzinfo is not None:
        raise ValueError("asof must be a timezone-naive market date.")
    d = d.loc[d["date"].le(stamp)].copy()
    if d.empty:
        raise ValueError("There are no observations on or before the requested date.")
    tickers = sorted(d["ticker"].unique())
    dates = pd.DatetimeIndex(sorted(set(d["date"]) | {stamp}), name="date")
    price = {
        col: d.pivot(index="date", columns="ticker", values=col).reindex(
            index=dates, columns=tickers
        )
        for col in ["spot", "call_mid", "put_mid"]
    }
    call, put = price["call_mid"], price["put_mid"]
    if basis == "Premium / spot":
        call = 100.0 * call.div(price["spot"])
        put = 100.0 * put.div(price["spot"])
    values = {"call": call, "put": put, "spread": call - put}
    scores, counts, means, stds, statuses = {}, {}, {}, {}, {}
    delta = pd.Timedelta(days=window_days)
    for metric, x in values.items():
        rolling = x.rolling(f"{window_days}D", closed="left", min_periods=2)
        mean, std = rolling.mean(), rolling.std(ddof=1)
        count = x.rolling(f"{window_days}D", closed="left", min_periods=0).count()
        mature = pd.DataFrame(False, index=dates, columns=tickers)
        for ticker in tickers:
            first = x[ticker].first_valid_index()
            if first is not None:
                mature[ticker] = dates >= first + delta
        eligible = count.ge(min_obs) & mature
        z = ((x - mean) / std.where(std.gt(1e-12))).where(eligible)
        scores[metric], counts[metric] = z, count
        means[metric], stds[metric] = mean, std
        status = pd.Series("OK", index=tickers, dtype=object)
        status.loc[~std.loc[stamp].gt(1e-12)] = "Zero/invalid standard deviation"
        status.loc[count.loc[stamp].lt(min_obs)] = f"Fewer than {min_obs} prior observations"
        status.loc[~mature.loc[stamp]] = f"Less than {window_days} days of history"
        status.loc[x.loc[stamp].isna()] = "No value at the as-of date"
        statuses[metric] = status

    latest = pd.DataFrame(index=pd.Index(tickers, name="Ticker"))
    latest["Spot"] = price["spot"].loc[stamp]
    latest["Call mid"] = price["call_mid"].loc[stamp]
    latest["Put mid"] = price["put_mid"].loc[stamp]
    for label, metric in METRICS.items():
        latest[label] = scores[metric].loc[stamp]
    quote_rows = d.loc[d[["call_mid", "put_mid"]].notna().any(axis=1)]
    last_quote = quote_rows.groupby("ticker")["date"].max()
    latest["Last option date"] = last_quote.reindex(tickers)
    return dict(
        latest=latest, scores=scores, values=values, price=price,
        counts=counts, means=means, stds=stds, statuses=statuses,
        asof=stamp, window_days=window_days, min_obs=min_obs, basis=basis,
    )

## 2.4 让公式在一张小表上真正跑一次

**以下完全是教学数据，不是股票行情。** 为了让六行数字就能看清边界，调用时临时使用 `window_days=5, min_obs=3`；原全局 `WINDOW_DAYS=365` 与 `MIN_OBSERVATIONS=200` 均不改动。

这些日期逐日排列，仅用于演示日历窗口，不代表市场交易日历。到 `2024-01-06` 时，5 天窗口是 `[2024-01-01, 2024-01-06)`；历史 Call 值是 1、2、3、4、5，今天的 6 不在基准中。

此前某些日期虽然已经有 3 个先前观察值，但从首个有效日期起尚未经过 5 天，所以还不能通过 `mature` 检查。下面的表会把数量、均值、标准差和 Z 一起列出来，方便你对照。

In [10]:
# 教学练习 6：缩短窗口只是为了看清计算，不改变仪表盘默认参数。
lesson_history = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=6, freq="D"),
    "ticker": ["DEMO_ONLY"] * 6,
    "spot": [100.0] * 6,
    "call_mid": [1., 2., 3., 4., 5., 6.],
    "put_mid": [2., 2., 3., 4., 4., 5.],
})
lesson_model = compute_dashboard(lesson_history, window_days=5, min_obs=3)
lesson_trace = pd.DataFrame({
    "当日Call": lesson_model["values"]["call"]["DEMO_ONLY"],
    "先前有效数量": lesson_model["counts"]["call"]["DEMO_ONLY"],
    "先前均值": lesson_model["means"]["call"]["DEMO_ONLY"],
    "先前样本标准差": lesson_model["stds"]["call"]["DEMO_ONLY"],
    "最终Call_Z": lesson_model["scores"]["call"]["DEMO_ONLY"],
})
display(lesson_trace)
print("最后一日的状态:", lesson_model["statuses"]["call"].loc["DEMO_ONLY"])
assert lesson_model["counts"]["call"].iloc[-1, 0] == 5
assert lesson_model["means"]["call"].iloc[-1, 0] == 3.0
assert np.isclose(lesson_model["scores"]["call"].iloc[-1, 0], 3 / np.sqrt(2.5))
assert lesson_model["scores"]["call"].iloc[:-1, 0].isna().all()
assert WINDOW_DAYS == 365 and MIN_OBSERVATIONS == 200

,当日Call,先前有效数量,先前均值,先前样本标准差,最终Call_Z
date,,,,,
2024-01-01,1.0,0.0,NaN,NaN,NaN
2024-01-02,2.0,1.0,NaN,NaN,NaN
2024-01-03,3.0,2.0,1.5,0.707107,NaN
2024-01-04,4.0,3.0,2.0,1.000000,NaN
2024-01-05,5.0,4.0,2.5,1.290994,NaN
2024-01-06,6.0,5.0,3.0,1.581139,1.897367


最后一日的状态: OK


In [11]:
# 教学练习 7：Spread Z 与 Call Z - Put Z 一般不同。
lesson_latest = lesson_model["latest"]
lesson_spread_z = lesson_latest.loc["DEMO_ONLY", "Spread Z"]
lesson_difference = (lesson_latest.loc["DEMO_ONLY", "Call Z"]
                     - lesson_latest.loc["DEMO_ONLY", "Put Z"])
display(lesson_latest[["Call Z", "Put Z", "Spread Z"]])
print("Spread Z:", round(lesson_spread_z, 6))
print("Call Z - Put Z:", round(lesson_difference, 6))
assert not np.isclose(lesson_spread_z, lesson_difference)

,Call Z,Put Z,Spread Z
Ticker,,,
DEMO_ONLY,1.897367,2.0,1.414214


Spread Z: 1.414214
Call Z - Put Z: -0.102633


In [12]:
# 教学练习 8：当日缺失不前向填充；历史标准差为 0 也不返回 Z=0。
lesson_missing = lesson_history.copy()
lesson_missing.loc[5, "call_mid"] = np.nan
lesson_missing_model = compute_dashboard(lesson_missing, window_days=5, min_obs=3)
print("当日 Call 缺失:", lesson_missing_model["statuses"]["call"].loc["DEMO_ONLY"])
assert pd.isna(lesson_missing_model["latest"].loc["DEMO_ONLY", "Call mid"])
assert pd.isna(lesson_missing_model["latest"].loc["DEMO_ONLY", "Call Z"])

lesson_constant = lesson_history.copy()
lesson_constant["call_mid"] = 2.0
lesson_constant_model = compute_dashboard(lesson_constant, window_days=5, min_obs=3)
print("Call 历史恒定:", lesson_constant_model["statuses"]["call"].loc["DEMO_ONLY"])
assert pd.isna(lesson_constant_model["latest"].loc["DEMO_ONLY", "Call Z"])

当日 Call 缺失: No value at the as-of date
Call 历史恒定: Zero/invalid standard deviation


## 2.5 返回字典里的每个名字都是什么？

**依据：原单元格 5 的返回语句。** 下表的“宽表”均是“日期 × 股票”的结构。

| 键 | 保存什么 | 主要被谁使用 |
|---|---|---|
| `latest` | 当前每只股票的股价、Call/Put 中间价、三个 Z 和最近任一报价日期 | 排名表 |
| `price` | `spot`、`call_mid`、`put_mid` 三张原始价格宽表 | 当前价格与下方价格图 |
| `values` | 按所选口径处理后的 Call、Put、Call−Put 宽表 | 统计计算 |
| `scores` | 三个指标各自的历史 Z 宽表 | 最新 Z 与上方 Z 图 |
| `counts` | 每个日期、每个指标、每只股票的先前有效值数量 | `Prior observations` |
| `means` / `stds` | 每个指标的历史滚动均值 / 样本标准差 | 计算与核对 |
| `statuses` | 每个指标在当前 `asof` 的逐股票状态；不是完整历史状态表 | `Z status` |
| `asof` | 当前统一展示日期 | 日期定位 |
| `window_days` / `min_obs` / `basis` | 本次计算采用的配置 | 结果说明与检查 |

`m["scores"]["call"].loc[某日, 某股票]` 可以读成：打开结果包 → 找到 Z-score 文件夹 → 找 Call 表 → 取某日、某股票交叉的那个格子。

<a id="retrieval"></a>
# 3. Direct BQL data retrieval and contract selection
## 数据怎样从 Bloomberg 进入这套计算？

**对应原单元格 6、7。** 这一节保留四个函数：

| 函数 | 一句话作用 | 是否直接调用 Bloomberg |
|---|---|---|
| `request_frame` | 发一次 BQL 请求，把多个字段对齐成表 | 是，调用时才请求 |
| `select_pair` | 从已拿到的期权链中挑一对 Call / Put | 否，只处理表 |
| `quote_midpoints` | 检查已拿到的 Bid / Ask，生成 Mid 和零买价标记 | 否，只处理表 |
| `fetch_bloomberg_history` | 按顺序组织成员、股价、选合约和报价历史请求 | 是，调用时才请求 |

**BQL 部分的解释边界：** 下面描述的是原代码打算请求什么、如何处理返回结果。公开的 Python / pandas 文档不能证明这些专有字段在你的 Bloomberg 账户中一定可用；原文件也明确保留了这一验证限制。

<details>
<summary>展开查看原文件第 6 个单元格的英文说明（原文保留）</summary>

## 3. Direct BQL data retrieval and contract selection

</details>

## 3.1 `request_frame`：把多个返回表可靠地拼在一起

**输入：** `bq` 是会话对象，`universe` 是查询的证券范围，`items` 是所需字段字典，`history` 决定是否按历史日期对齐。  
**输出：** 一张以 `ID` 或 `ID + DATE` 为对齐键的表。

BQL 的各个字段结果先各自转成表，再按标识和日期合并。不能仅因为两张表行数相同，就认定它们同一行一定对应同一证券、同一日期。

### 3.1.1 创建请求、执行请求，再逐个处理字段

**原文件定位：第 7 个单元格，第 1–13 行。以下是阅读片段，不要单独运行。**

```python
def request_frame(bq, universe, items: dict, history: bool = False) -> pd.DataFrame:
    """Join named BQL results explicitly; reject ambiguous observations."""
    import bql
    response = bq.execute(bql.Request(universe, items))
    if len(response) != len(items):
        raise ValueError("BQL returned a different number of fields than requested.")
    keys = ["ID", "DATE"] if history else ["ID"]
    pieces = []
    for name, result in zip(items, response):
        frame = result.df().reset_index()
        if frame.empty:
            pieces.append(pd.DataFrame(columns=keys + [name]))
            continue
```

`import bql` 写在函数内部，所以**定义这个函数时还不会导入 bql**。只有调用函数，才会走到这里。

`bql.Request(universe, items)` 组装请求；`bq.execute(...)` 才实际执行。`len(response) != len(items)` 检查返回的字段结果数量是否与请求数量相同，不是检查股票数量。

历史请求用 `keys = ["ID", "DATE"]`，非历史请求只用 `["ID"]`。`zip(items, response)` 将字段名称与返回对象按迭代顺序配对；后面还要求结果表中存在同名字段列，避免把错列直接当成需要的数据。

`result.df().reset_index()` 转成表，并把索引恢复成普通列。空结果会被整理成“有列名、无数据”的表加入列表，不会凭空补值。`continue` 表示本轮结束，继续处理下一个字段。

### 3.1.2 检查列名、币种、标识和历史日期

**原文件定位：第 7 个单元格，第 14–30 行。以下是阅读片段，不要单独运行。**

```python
        missing = set(keys + [name]).difference(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing BQL columns {sorted(missing)}; got {list(frame.columns)}")
        if "CURRENCY" in frame and history:
            currencies = set(frame["CURRENCY"].dropna().astype(str).str.upper())
            if currencies - {"USD"}:
                raise ValueError(f"{name}: expected USD prices; got {currencies}.")
        frame = frame[keys + [name]].copy()
        if frame["ID"].isna().any():
            raise ValueError(f"{name}: BQL returned a missing security ID.")
        frame["ID"] = frame["ID"].astype(str)
        if history:
            frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise")
            if frame["DATE"].isna().any() or frame["DATE"].dt.tz is not None:
                raise ValueError(f"{name}: expected timezone-naive market dates.")
            frame["DATE"] = frame["DATE"].dt.normalize()
            frame[name] = pd.to_numeric(frame[name], errors="raise").replace([np.inf, -np.inf], np.nan)
```

`set(keys + [name]).difference(frame.columns)` 检查对齐键和目标字段是否都存在。例如历史 Bid 结果需要 `ID`、`DATE`、`Bid`。

如果历史返回表带有 `CURRENCY` 列，程序把其中非缺失的币种转成大写；发现不是 USD 的值就报错。**这不是汇率换算，也不代表已经验证所有数据：没有币种列或币种全缺失时，这段检查无法证明报价口径是 USD。**

`frame = frame[keys + [name]].copy()` 排除本次不需要的额外维度，但随后仍要求指定键唯一。证券 ID 不允许缺失，之后转成字符串。

历史日期必须可转换、不缺失、不带时区，并归到当日零点。历史数值中的正负无穷会先替换成 `NaN`，与后面直接输入 `validate_history` 时遇到无穷值就报错的处理不同：它们位于不同的数据入口阶段。

### 3.1.3 拒绝重复键，然后做一对一外连接

**原文件定位：第 7 个单元格，第 31–37 行。以下是阅读片段，不要单独运行。**

```python
        if frame.duplicated(keys).any():
            raise ValueError(f"{name}: duplicate {keys}; inspect returned BQL dimensions.")
        pieces.append(frame)
    return reduce(
        lambda a, b: a.merge(b, on=keys, how="outer", validate="one_to_one"),
        pieces,
    )
```

若某个字段有重复的 `ID + DATE`（或非历史请求的 `ID`），代码报错，让使用者检查返回维度，而不是随便取其中一行。

`reduce(lambda a, b: ..., pieces)` 依次合并列表中的表：先合并前两个，再把合并结果与第三个合并，以此类推。`lambda a, b: ...` 是一个简短的匿名函数。

`how="outer"` 是外连接：保留两边出现过的全部键，没有匹配的一侧留空。`validate="one_to_one"` 再次要求两边键都是唯一的，避免重复匹配把数据意外膨胀。

此函数的调用方都提供非空字段字典；原实现没有为“请求零个字段”的情况单独设计返回值。

### 运行完整函数：`request_frame`

下一格只是定义函数；默认的离线教学流程不会调用它，也不会创建伪装成 Bloomberg 的服务。

In [13]:
def request_frame(bq, universe, items: dict, history: bool = False) -> pd.DataFrame:
    """Join named BQL results explicitly; reject ambiguous observations."""
    import bql
    response = bq.execute(bql.Request(universe, items))
    if len(response) != len(items):
        raise ValueError("BQL returned a different number of fields than requested.")
    keys = ["ID", "DATE"] if history else ["ID"]
    pieces = []
    for name, result in zip(items, response):
        frame = result.df().reset_index()
        if frame.empty:
            pieces.append(pd.DataFrame(columns=keys + [name]))
            continue
        missing = set(keys + [name]).difference(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing BQL columns {sorted(missing)}; got {list(frame.columns)}")
        if "CURRENCY" in frame and history:
            currencies = set(frame["CURRENCY"].dropna().astype(str).str.upper())
            if currencies - {"USD"}:
                raise ValueError(f"{name}: expected USD prices; got {currencies}.")
        frame = frame[keys + [name]].copy()
        if frame["ID"].isna().any():
            raise ValueError(f"{name}: BQL returned a missing security ID.")
        frame["ID"] = frame["ID"].astype(str)
        if history:
            frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise")
            if frame["DATE"].isna().any() or frame["DATE"].dt.tz is not None:
                raise ValueError(f"{name}: expected timezone-naive market dates.")
            frame["DATE"] = frame["DATE"].dt.normalize()
            frame[name] = pd.to_numeric(frame[name], errors="raise").replace([np.inf, -np.inf], np.nan)
        if frame.duplicated(keys).any():
            raise ValueError(f"{name}: duplicate {keys}; inspect returned BQL dimensions.")
        pieces.append(frame)
    return reduce(
        lambda a, b: a.merge(b, on=keys, how="outer", validate="one_to_one"),
        pieces,
    )

In [14]:
# 教学练习 9：仅用本地表模拟“按键合并”的动作，不模拟 BQL 服务。
lesson_bid_table = pd.DataFrame({
    "ID": ["TEACH_CONTRACT", "TEACH_CONTRACT"],
    "DATE": pd.to_datetime(["2024-01-01", "2024-01-02"]),
    "Bid": [2.0, 2.2],
})
lesson_ask_table = pd.DataFrame({
    "ID": ["TEACH_CONTRACT", "TEACH_CONTRACT"],
    "DATE": pd.to_datetime(["2024-01-02", "2024-01-03"]),
    "Ask": [2.4, 2.6],
})
lesson_joined = lesson_bid_table.merge(
    lesson_ask_table, on=["ID", "DATE"], how="outer", validate="one_to_one"
).sort_values("DATE")
display(lesson_joined)
print("只有 1 月 2 日两边都出现；其余日期保留缺失，而不是按行号拼错。")
assert len(lesson_joined) == 3
assert lesson_joined["Bid"].isna().sum() == 1
assert lesson_joined["Ask"].isna().sum() == 1

,ID,DATE,Bid,Ask
0,TEACH_CONTRACT,2024-01-01,2.0,NaN
1,TEACH_CONTRACT,2024-01-02,2.2,2.4
2,TEACH_CONTRACT,2024-01-03,NaN,2.6


只有 1 月 2 日两边都出现；其余日期保留缺失，而不是按行号拼错。


## 3.2 `select_pair`：究竟怎样选出当前的期权对？

**输入：** 当前期权链表 `chain`、当前股价 `spot`、统一市场日期 `asof`、目标 DTE 和容差。  
**输出：** 一个字典，保存到期日、行权价、DTE、Call 合约 ID、Put 合约 ID。

它选择的是**同一到期日、同一行权价的一对**，不是各自挑一个看上去最接近的 Call 和 Put。

### 3.2.1 清理期权链，并按到期天数筛选

**原文件定位：第 7 个单元格，第 40–61 行。以下是阅读片段，不要单独运行。**

```python
def select_pair(chain: pd.DataFrame, spot: float, asof: pd.Timestamp,
                target_dte: int, tolerance: int) -> dict:
    """Closest eligible expiry first, then closest strike with one call and put.

    Reject duplicate expiry/strike/side records rather than silently choosing an
    exchange or adjusted contract. Unique pairs still require deliverable review.
    """
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("No valid underlying price on the common as-of date.")
    c = chain.copy()
    c["Expiry"] = pd.to_datetime(c["Expiry"], errors="coerce").dt.normalize()
    c["Strike"] = pd.to_numeric(c["Strike"], errors="coerce")
    c["Side"] = c["Side"].astype(str).str.strip().str.upper().map(
        {"CALL": "C", "C": "C", "PUT": "P", "P": "P"}
    )
    c["DTE"] = (c["Expiry"] - asof).dt.days
    c = c.loc[c["Side"].notna() & c["Strike"].gt(0)
              & c["DTE"].ge(MIN_DTE)
              & c["DTE"].sub(target_dte).abs().le(tolerance)].copy()
    c = c.loc[~c.duplicated(["Expiry", "Strike", "Side"], keep=False)]
    if c.empty:
        raise ValueError("No unambiguous option pair within the requested expiry band.")
```

先要求当前股价是有限的正数。复制期权链后，把到期日转成日期、行权价转成数字。这里 `errors="coerce"` 与前面的 `errors="raise"` 不同：转换失败先变成缺失，随后在筛选中排除。

Call / Put 字段去空白、转大写后，统一映射成 `C` / `P`；无法识别的类别变成缺失。`DTE = (Expiry - asof).dt.days` 计算日历天差。

随后要求：类别有效、行权价大于 0、DTE 至少为 `MIN_DTE`、且偏离目标不超过容差。这个阶段没有按成交量、未平仓量、Bid / Ask 或报价时间筛选。

`duplicated(["Expiry", "Strike", "Side"], keep=False)` 将同一到期日、行权价、方向下的**全部重复记录**标记出来，前面的 `~` 反转标记，只保留不重复的记录。不是留第一条，也不是只删除多出来的一条。

**精确地说：** 出现重复不一定让整只股票失败。受影响的候选记录被移除；只要其他地方仍有合法配对，就可以继续选择。若剩余表为空才报错。

### 3.2.2 配对之后，按照四层排序规则选第一行

**原文件定位：第 7 个单元格，第 62–72 行。以下是阅读片段，不要单独运行。**

```python
    pairs = c.pivot(index=["Expiry", "Strike", "DTE"], columns="Side", values="ID")
    if not {"C", "P"}.issubset(pairs.columns):
        raise ValueError("No matched call and put in the eligible chain.")
    pairs = pairs.dropna(subset=["C", "P"]).reset_index()
    if pairs.empty:
        raise ValueError("No call and put share an eligible expiry and strike.")
    pairs["expiry_distance"] = pairs["DTE"].sub(target_dte).abs()
    pairs["strike_distance"] = pairs["Strike"].sub(spot).abs()
    p = pairs.sort_values(["expiry_distance", "Expiry", "strike_distance", "Strike"]).iloc[0]
    return {"Expiry": p["Expiry"], "Strike": float(p["Strike"]),
            "DTE": int(p["DTE"]), "Call contract": p["C"], "Put contract": p["P"]}
```

`pivot` 把 `C` 和 `P` 变成两列，用到期日、行权价、DTE 标识同一组。先要求存在两类列，再删除任意一边缺失的行；因此留下的每一行都有一张 Call 和一张 Put。

之后计算两个绝对距离：`expiry_distance = abs(DTE - target_dte)`，`strike_distance = abs(Strike - spot)`。

**原代码完整的优先顺序是：**

1. 距目标 DTE 更近的排在前面。
2. 若到期距离相同，先选更早的到期日。
3. 在已确定的到期日内，先选更接近股价的行权价。
4. 若行权价距离仍相同，先选数值更小的行权价。

`.iloc[0]` 取排完序后的第一行。`float(...)` 与 `int(...)` 将返回值整理成明确的数值类型。

**由代码可见的限制：** 此处只核对日期、行权价和方向，不能据此断言两张合约的调整状态或交割内容一致；也没有保证最终选中的合约历史足够长、报价足够完整。代码不会因它后来没有 Z-score 就自动改选另一张。

### 运行完整函数：`select_pair`

这个函数不依赖 Bloomberg 连接，因此可以用下面的虚构期权链观察选择规则。

In [15]:
def select_pair(chain: pd.DataFrame, spot: float, asof: pd.Timestamp,
                target_dte: int, tolerance: int) -> dict:
    """Closest eligible expiry first, then closest strike with one call and put.

    Reject duplicate expiry/strike/side records rather than silently choosing an
    exchange or adjusted contract. Unique pairs still require deliverable review.
    """
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("No valid underlying price on the common as-of date.")
    c = chain.copy()
    c["Expiry"] = pd.to_datetime(c["Expiry"], errors="coerce").dt.normalize()
    c["Strike"] = pd.to_numeric(c["Strike"], errors="coerce")
    c["Side"] = c["Side"].astype(str).str.strip().str.upper().map(
        {"CALL": "C", "C": "C", "PUT": "P", "P": "P"}
    )
    c["DTE"] = (c["Expiry"] - asof).dt.days
    c = c.loc[c["Side"].notna() & c["Strike"].gt(0)
              & c["DTE"].ge(MIN_DTE)
              & c["DTE"].sub(target_dte).abs().le(tolerance)].copy()
    c = c.loc[~c.duplicated(["Expiry", "Strike", "Side"], keep=False)]
    if c.empty:
        raise ValueError("No unambiguous option pair within the requested expiry band.")
    pairs = c.pivot(index=["Expiry", "Strike", "DTE"], columns="Side", values="ID")
    if not {"C", "P"}.issubset(pairs.columns):
        raise ValueError("No matched call and put in the eligible chain.")
    pairs = pairs.dropna(subset=["C", "P"]).reset_index()
    if pairs.empty:
        raise ValueError("No call and put share an eligible expiry and strike.")
    pairs["expiry_distance"] = pairs["DTE"].sub(target_dte).abs()
    pairs["strike_distance"] = pairs["Strike"].sub(spot).abs()
    p = pairs.sort_values(["expiry_distance", "Expiry", "strike_distance", "Strike"]).iloc[0]
    return {"Expiry": p["Expiry"], "Strike": float(p["Strike"]),
            "DTE": int(p["DTE"]), "Call contract": p["C"], "Put contract": p["P"]}

In [16]:
# 教学练习 10：这些 ID 都是虚构标签，绝不发送给 Bloomberg。
lesson_asof = pd.Timestamp("2024-01-01")
lesson_chain_rows = []
for lesson_dte, lesson_strike in [(29, 100), (30, 98), (30, 102), (60, 100)]:
    for lesson_side in ["C", "P"]:
        lesson_chain_rows.append({
            "ID": f"TEACH_{lesson_side}_{lesson_dte}_{lesson_strike}",
            "Expiry": lesson_asof + pd.Timedelta(days=lesson_dte),
            "Strike": lesson_strike,
            "Side": lesson_side,
        })
lesson_chain = pd.DataFrame(lesson_chain_rows)
lesson_pair = select_pair(lesson_chain, spot=100.0, asof=lesson_asof,
                          target_dte=30, tolerance=14)
display(pd.DataFrame([lesson_pair]))
print("先选择 30 天，而不是 29 天；98 和 102 距 100 相同，所以选 98。")
assert lesson_pair["DTE"] == 30 and lesson_pair["Strike"] == 98.0

# 再人为添加一条重复的 30 天、98 行权价的 Call。
lesson_extra = lesson_chain.loc[lesson_chain["ID"].eq("TEACH_C_30_98")].copy()
lesson_extra["ID"] = "TEACH_DUPLICATE_CALL"
lesson_chain_duplicate = pd.concat([lesson_chain, lesson_extra], ignore_index=True)
lesson_pair_after_duplicate = select_pair(
    lesson_chain_duplicate, spot=100.0, asof=lesson_asof, target_dte=30, tolerance=14
)
print("出现重复 Call 后，两条重复 Call 都被排除，改选：", lesson_pair_after_duplicate["Strike"])
assert lesson_pair_after_duplicate["Strike"] == 102.0

,Expiry,Strike,DTE,Call contract,Put contract
0,2024-01-31,98.0,30,TEACH_C_30_98,TEACH_P_30_98


先选择 30 天，而不是 29 天；98 和 102 距 100 相同，所以选 98。
出现重复 Call 后，两条重复 Call 都被排除，改选： 102.0


## 3.3 `quote_midpoints`：中间价不只是直接加起来除以二

**输入：** 含 Bid、Ask 的报价表。  
**输出：** 原表副本，新增 `Mid` 与 `Zero bid` 两列。  
**依据：原单元格 7 第 75–82 行。**

只有两边都是有限数，且同时满足 `Bid >= 0`、`Ask > 0`、`Ask >= Bid` 时，才使用 `Mid = (Bid + Ask) / 2`。

不满足条件的行**保留在表里，但 Mid 设为缺失**；不是删除整行。`Zero bid` 只对符合上述条件且 Bid 恰为 0 的记录标为 True。

Ask 低于 Bid 的交叉报价被拒绝。Bid 为 0、Ask 大于 0 的报价保留并标记；Bid 与 Ask 同为正数且相等也会通过，因为代码允许 `Ask == Bid`。这段逻辑没有另外限制买卖价差宽度，也没有验证两边报价的具体时间戳。

In [17]:
def quote_midpoints(quotes: pd.DataFrame) -> pd.DataFrame:
    """Midpoints only where both quotes exist and ask >= bid >= 0, ask > 0."""
    q = quotes.copy()
    valid = (np.isfinite(q["Bid"]) & np.isfinite(q["Ask"])
             & q["Bid"].ge(0) & q["Ask"].gt(0) & q["Ask"].ge(q["Bid"]))
    q["Mid"] = ((q["Bid"] + q["Ask"]) / 2.0).where(valid)
    q["Zero bid"] = valid & q["Bid"].eq(0)
    return q

In [18]:
# 教学练习 11：用边界情况直接看过滤结果。
lesson_quotes = pd.DataFrame({
    "情况": ["正常", "零买价", "交叉报价", "负买价", "双零", "买价缺失", "无穷卖价", "正数同价"],
    "Bid": [2.0, 0.0, 3.0, -0.1, 0.0, np.nan, 1.0, 2.0],
    "Ask": [2.4, 0.4, 2.0, 0.2, 0.0, 2.0, np.inf, 2.0],
})
lesson_midpoints = quote_midpoints(lesson_quotes)
display(lesson_midpoints)
assert lesson_midpoints["Mid"].notna().tolist() == [True, True, False, False, False, False, False, True]
assert lesson_midpoints["Zero bid"].tolist() == [False, True, False, False, False, False, False, False]
assert np.isclose(lesson_midpoints.loc[0, "Mid"], 2.2)

,情况,Bid,Ask,Mid,Zero bid
0,正常,2.0,2.4,2.2,False
1,零买价,0.0,0.4,0.2,True
2,交叉报价,3.0,2.0,NaN,False
3,负买价,-0.1,0.2,NaN,False
4,双零,0.0,0.0,NaN,False
5,买价缺失,NaN,2.0,NaN,False
6,无穷卖价,1.0,inf,NaN,False
7,正数同价,2.0,2.0,2.0,False


## 3.4 `fetch_bloomberg_history`：整个取数流程的总调度

**输入：** 会话 `bq`、进度通知函数，以及当前的选合约设置。  
**输出：** `history`、`contracts`、`errors`、`asof`、`target_dte`、`tolerance` 组成的字典。

它在原文件里有 123 行。不要从头死记，先抓住这个顺序：

```text
校验参数 → 取成员 → 取股价 → 定 asof → 查期权链并选对
       → 取选中合约的 Bid/Ask → 算 Mid → 按股票与日期合并 → 返回
```

接下来的各段是完整原函数的阅读切片；最后才给出可一次运行的完整函数定义。这里不会实际执行 Bloomberg 请求。

### 3.4.1 默认参数、回调函数和参数检查

**原文件定位：第 7 个单元格，第 85–99 行。以下是阅读片段，不要单独运行。**

```python
def fetch_bloomberg_history(bq, progress: Callable[[str], None] = lambda _: None,
                           target_dte: int = TARGET_DTE,
                           tolerance: int = EXPIRY_TOLERANCE_DAYS,
                           max_stocks: Optional[int] = MAX_STOCKS) -> dict:
    """Fetch current SPX members, selected listed contracts, and daily prices.

    The '0D' endpoint requests the latest available daily observation; this is
    not a streaming subscription or a guarantee of an up-to-the-second quote.
    No forward filling, theoretical-pricing fallback, or replacement data source.
    Confirm BQL fields, historical bid/ask coverage, and entitlements in BQNT.
    """
    if target_dte < MIN_DTE or tolerance < 0:
        raise ValueError(f"Target DTE must be >= {MIN_DTE}; tolerance must be >= 0.")
    if max_stocks is not None and max_stocks < 1:
        raise ValueError("MAX_STOCKS must be None or a positive integer.")
```

`progress` 预期是一个“接收字符串、发出进度提示”的函数。默认的 `lambda _: None` 收到文字后什么也不做，所以取数函数即使不接界面，也不需要到处判断有没有进度条。

`Callable[[str], None]` 是对此的类型提示。后面每次 `progress("...")` 只是调用传入的通知函数，不是另起线程。

目标 DTE 不得低于 `MIN_DTE`，容差不得小于 0；股票数量上限要么是 `None`，要么至少为 1。这些检查按代码写出的数值关系执行，并没有为所有可能的错误类型逐项提供定制信息。

注释中的 `0D` 表达请求最新可得的日度端点，不是实时订阅。原程序没有历史不足时的理论定价补偿或替代数据源。

### 3.4.2 取当前成分股，分批取股价，确定 asof

**原文件定位：第 7 个单元格，第 100–123 行。以下是阅读片段，不要单独运行。**

```python
    progress("Retrieving current SPX Index members...")
    members = request_frame(bq, bq.univ.members(INDEX), {"Name": bq.data.name()})
    if members.empty:
        raise ValueError("Bloomberg returned no current index constituents.")
    members = members.sort_values("ID").set_index("ID")
    if max_stocks is not None:
        members = members.head(max_stocks)
    tickers = members.index.tolist()

    # Use date-stamped underlying history to establish one common market date.
    dates = bq.func.range("-2Y", "0D")
    spot_parts = []
    for i in range(0, len(tickers), BATCH_SIZE):
        progress(f"Underlying history: {min(i+BATCH_SIZE, len(tickers))}/{len(tickers)} stocks")
        spot_parts.append(request_frame(
            bq, tickers[i:i+BATCH_SIZE],
            {"spot": bq.data.px_last(dates=dates, fill="NA")}, history=True,
        ))
    spots = pd.concat(spot_parts, ignore_index=True)
    spots = spots.loc[spots["spot"].gt(0)].copy()
    if spots.empty:
        raise ValueError("No valid underlying history was returned.")
    asof = pd.Timestamp(spots["DATE"].max())
    current_spot = spots.loc[spots["DATE"].eq(asof)].set_index("ID")["spot"]
```

`bq.univ.members(INDEX)` 表达指数成员范围；`bq.data.name()` 请求名称。`request_frame` 将其整理成表。成员为空则停止；否则按 ID 排序、设为索引，并在设置了上限时取前若干个。这里不是随机抽样或市值排名。

`bq.func.range("-2Y", "0D")` 表达原代码请求约两年历史的意图。`range(0, len(tickers), BATCH_SIZE)` 给出 0、40、80……的批次起点，切片 `tickers[i:i+BATCH_SIZE]` 取当前批次。

`px_last(..., fill="NA")` 请求日度股价历史并要求缺失不要向前填充。**请求意图不等于已验证服务端语义**，实际字段、填充与权限仍需在 BQNT 检查。`pd.concat(..., ignore_index=True)` 将各批结果按行接在一起，只留下股价大于 0 的观察值。

`asof` 是所有保留下来的股价记录中的最大日期。它是统一使用的展示日期，**不是所有股票都必然有数据的日期交集**。`current_spot` 只保存这一天确实有股价的股票；没有该日股价的股票稍后不能选合约。

为什么请求约两年？当前 Z 需要往前一年；若还想画过去一年的滚动 Z，图中较早日期也需要更早的基准历史。但请求区间长，不代表当前选中的合约上市已足够久。

### 3.4.3 逐只股票取当前期权链、选择合约并记录问题

**原文件定位：第 7 个单元格，第 124–149 行。以下是阅读片段，不要单独运行。**

```python
    meta = members.rename_axis("Ticker").copy()
    meta["Selection status"] = "Not processed"
    errors = []
    pairs = []
    chain_items = {"Expiry": bq.data.expire_dt(),
                   "Strike": bq.data.strike_px(), "Side": bq.data.put_call()}
    completed_chain_requests = 0
    failed_chain_requests = 0
    for i, ticker in enumerate(tickers, 1):
        progress(f"Selecting listed call/put pairs: {i}/{len(tickers)} — {ticker}")
        try:
            if ticker not in current_spot:
                raise ValueError("No underlying price on the common as-of date.")
            try:
                chain = request_frame(bq, bq.univ.options(ticker), chain_items)
            except Exception:
                failed_chain_requests += 1
                raise
            completed_chain_requests += 1
            selected = select_pair(chain, float(current_spot[ticker]), asof, target_dte, tolerance)
            pairs.append({"Ticker": ticker, **selected})
            meta.loc[ticker, "Selection status"] = "OK"
        except Exception as exc:
            message = str(exc)
            meta.loc[ticker, "Selection status"] = message
            errors.append({"Stage": "Option selection", "Security": ticker, "Error": message})
```

`meta` 从成分股名单开始，索引名改成 `Ticker`，随后加入合约与状态信息。`pairs` 保存选中的期权对，`errors` 保存问题。`chain_items` 表达所需的到期日、行权价和 Call/Put 方向字段。

`enumerate(tickers, 1)` 同时给出从 1 开始的进度序号和股票标识。先检查该股票在共同日期有没有股价，再请求 `bq.univ.options(ticker)` 的当前期权链。**这一阶段逐只股票查询，不是每 40 只一批。**

拿到链之后调用 `select_pair(...)`。`{"Ticker": ticker, **selected}` 中的 `**selected` 把选中结果字典展开，合成包含股票标识的一条记录。成功状态写为 `OK`。

`except Exception as exc` 捕捉这一只股票的取链或选择错误，将原因写入 `meta` 和 `errors`。因此单只股票找不到合适配对时，通常可以继续处理其他股票，而不是隐藏问题。

### 3.4.4 哪些期权链错误会让整个流程提前停止？

**原文件定位：第 7 个单元格，第 150–158 行。以下是阅读片段，不要单独运行。**

```python
            # Avoid hundreds of identical queries after an API/schema failure.
            if completed_chain_requests == 0 and failed_chain_requests >= 3:
                raise RuntimeError("The first option-chain requests failed. " + message) from exc
    if pairs:
        selected = pd.DataFrame(pairs).set_index("Ticker")
        meta = meta.join(selected)
    else:
        for col in ["Expiry", "Strike", "DTE", "Call contract", "Put contract"]:
            meta[col] = np.nan
```

计数器区分“期权链请求本身成功完成”和“成功选出配对”。即使链请求返回成功但链里没有合适配对，它仍计入 `completed_chain_requests`。

若**还没有任何一次期权链请求成功完成**，但已有至少三次链请求失败，代码直接抛出 `RuntimeError`，避免在明显的 API 或字段问题下继续重复请求很多股票。它不是“每出现三次错误就停止”，也不是一般性的自动重试。

如果至少选中一对，就把 `pairs` 转成表，按股票索引合并到 `meta`。若一对都没有，仍建立到期日、行权价与合约 ID 列，值保持缺失。程序不会在这里编造期权对。

### 3.4.5 只查询已选合约的 Bid / Ask，并区分首批与后续失败

**原文件定位：第 7 个单元格，第 160–181 行。以下是阅读片段，不要单独运行。**

```python
    option_ids = sorted(set(meta["Call contract"].dropna()) | set(meta["Put contract"].dropna()))
    history_dates = bq.func.range("-2Y", asof.strftime("%Y-%m-%d"))
    # Concrete BQL data items: these are listed-option bid/ask prices, not IV.
    quote_items = {"Bid": bq.data.px_bid(dates=history_dates, fill="NA"),
                   "Ask": bq.data.px_ask(dates=history_dates, fill="NA")}
    quote_parts = []
    failed_option_ids = set()
    for i in range(0, len(option_ids), BATCH_SIZE):
        batch = option_ids[i:i+BATCH_SIZE]
        progress(f"Option bid/ask history: {min(i+BATCH_SIZE, len(option_ids))}/{len(option_ids)} contracts")
        try:
            quote_parts.append(request_frame(bq, batch, quote_items, history=True))
        except Exception as exc:
            if i == 0:
                raise RuntimeError(
                    "First historical PX_BID/PX_ASK request failed; check these "
                    "items and their history entitlement in BQNT. " + str(exc)
                ) from exc
            failed_option_ids.update(batch)
            errors.append({"Stage": "Option history", "Security": ", ".join(batch), "Error": str(exc)})
    quotes = (quote_midpoints(pd.concat(quote_parts, ignore_index=True))
              if quote_parts else pd.DataFrame(columns=["ID", "DATE", "Bid", "Ask", "Mid", "Zero bid"]))
```

`option_ids` 是 Call 与 Put 合约 ID 集合的并集：去掉缺失、去重，再排序。之后按最多 40 个合约一批请求。这里的“40”计数单位是期权合约，不是 40 对，也不一定等于 40 只股票。

`history_dates` 的结束日期明确写成共同 `asof`。字段是 `px_bid` 与 `px_ask`，不是隐含波动率。代码没有每个历史日期重查期权链；固定的合约 ID 会用于整段历史请求。

如果第一批报价历史请求失败，整个加载停止，并提示检查历史 PX_BID / PX_ASK 字段和权限。后续批次失败时，把**该批所有合约**加入失败集合、把错误记入列表，再继续其余批次。它不会逐张拆分重试。

成功返回的批次连接起来，交给 `quote_midpoints`。若没有报价结果，则建立带预期列名的空表，而不是插入随机行情。

### 3.4.6 以合约 ID 找报价，再按股票和日期对齐

**原文件定位：第 7 个单元格，第 182–198 行。以下是阅读片段，不要单独运行。**

```python
    by_id = {key: group for key, group in quotes.groupby("ID")}
    by_stock = {key: group for key, group in spots.groupby("ID")}
    rows = []
    for ticker in tickers:
        spot = by_stock.get(ticker, pd.DataFrame(columns=["DATE", "spot"]))
        panel = spot[["DATE", "spot"]].set_index("DATE")
        for side, col in [("Call", "call_mid"), ("Put", "put_mid")]:
            option_id = meta.loc[ticker, f"{side} contract"]
            q = by_id.get(option_id, pd.DataFrame(columns=["DATE", "Mid", "Zero bid"]))
            series = q.set_index("DATE")["Mid"].rename(col)
            panel = panel.join(series, how="outer", validate="one_to_one")
            valid_dates = q.loc[q["Mid"].notna(), "DATE"]
            meta.loc[ticker, f"{side} last date"] = valid_dates.max() if not valid_dates.empty else pd.NaT
            on_date = q.loc[q["DATE"].eq(asof)]
            meta.loc[ticker, f"{side} zero bid"] = bool(on_date["Zero bid"].any())
            if option_id in failed_option_ids:
                meta.loc[ticker, "Selection status"] = "Option-history request failed; see query errors"
```

`quotes.groupby("ID")` 与 `spots.groupby("ID")` 分别按期权合约和股票分组；两个字典便于稍后通过 ID 查到对应的整段历史。

对每只股票，先以它的股价日期为索引建立 `panel`。再分别取当前已选 Call 和 Put 的固定合约 ID，用该合约的 `Mid` 历史生成 `call_mid` / `put_mid` 列，按日期做一对一外连接。

`.get(key, 空表)` 表示找不到时提供结构正确的空对象，以便下游仍可保留这只股票，而不是把不存在的数据当成价格 0。

`Call last date` / `Put last date` 是各自 Mid 非缺失的最后日期。`Call zero bid` / `Put zero bid` 则只查看**共同 asof 当天**是否有被接受的零买价记录；它们不是“历史上从没出现过零买价”的证明，False 也可能只是当日没有合格报价。

如果合约属于失败的后续报价批次，会将该股票状态改成“期权历史请求失败”，使界面能显示取数问题。

### 3.4.7 合并所有股票，返回取数结果包

**原文件定位：第 7 个单元格，第 199–207 行。以下是阅读片段，不要单独运行。**

```python
        panel = panel.reindex(panel.index.union(pd.DatetimeIndex([asof])))
        panel.index.name = "date"
        panel["ticker"] = ticker
        for col in ["spot", "call_mid", "put_mid"]:
            panel[col] = pd.to_numeric(panel[col], errors="raise").astype(float)
        rows.append(panel.reset_index())
    raw = validate_history(pd.concat(rows, ignore_index=True))
    return {"history": raw, "contracts": meta, "errors": pd.DataFrame(errors),
            "asof": asof, "target_dte": target_dte, "tolerance": tolerance}
```

每只股票的日期索引都会加入 `asof`，即使那天期权价格缺失也保留一行。索引命名为 `date`，加入 `ticker` 列，再把价格列转成浮点数。

将所有股票的面板按行连接，最后调用 `validate_history`。输出的 `history` 就是计算函数需要的五列长表。

`contracts` 保存股票名称、所选合约、到期日、行权价、DTE、状态与最近报价日期；`errors` 是问题记录表；其余字段说明这次取数使用的日期与目标区间。

**注意阅读片段边界：** 这里引用的是函数末尾，包含面板索引整理与最终返回语句。没有省略或改写原实现。

### 运行完整函数：`fetch_bloomberg_history`

下一格完整保留原单元格 7 第 85–207 行。**执行本格仅定义函数，不会取数。** 真正的调用发生在用户点击加载按钮之后。

In [19]:
def fetch_bloomberg_history(bq, progress: Callable[[str], None] = lambda _: None,
                           target_dte: int = TARGET_DTE,
                           tolerance: int = EXPIRY_TOLERANCE_DAYS,
                           max_stocks: Optional[int] = MAX_STOCKS) -> dict:
    """Fetch current SPX members, selected listed contracts, and daily prices.

    The '0D' endpoint requests the latest available daily observation; this is
    not a streaming subscription or a guarantee of an up-to-the-second quote.
    No forward filling, theoretical-pricing fallback, or replacement data source.
    Confirm BQL fields, historical bid/ask coverage, and entitlements in BQNT.
    """
    if target_dte < MIN_DTE or tolerance < 0:
        raise ValueError(f"Target DTE must be >= {MIN_DTE}; tolerance must be >= 0.")
    if max_stocks is not None and max_stocks < 1:
        raise ValueError("MAX_STOCKS must be None or a positive integer.")
    progress("Retrieving current SPX Index members...")
    members = request_frame(bq, bq.univ.members(INDEX), {"Name": bq.data.name()})
    if members.empty:
        raise ValueError("Bloomberg returned no current index constituents.")
    members = members.sort_values("ID").set_index("ID")
    if max_stocks is not None:
        members = members.head(max_stocks)
    tickers = members.index.tolist()

    # Use date-stamped underlying history to establish one common market date.
    dates = bq.func.range("-2Y", "0D")
    spot_parts = []
    for i in range(0, len(tickers), BATCH_SIZE):
        progress(f"Underlying history: {min(i+BATCH_SIZE, len(tickers))}/{len(tickers)} stocks")
        spot_parts.append(request_frame(
            bq, tickers[i:i+BATCH_SIZE],
            {"spot": bq.data.px_last(dates=dates, fill="NA")}, history=True,
        ))
    spots = pd.concat(spot_parts, ignore_index=True)
    spots = spots.loc[spots["spot"].gt(0)].copy()
    if spots.empty:
        raise ValueError("No valid underlying history was returned.")
    asof = pd.Timestamp(spots["DATE"].max())
    current_spot = spots.loc[spots["DATE"].eq(asof)].set_index("ID")["spot"]
    meta = members.rename_axis("Ticker").copy()
    meta["Selection status"] = "Not processed"
    errors = []
    pairs = []
    chain_items = {"Expiry": bq.data.expire_dt(),
                   "Strike": bq.data.strike_px(), "Side": bq.data.put_call()}
    completed_chain_requests = 0
    failed_chain_requests = 0
    for i, ticker in enumerate(tickers, 1):
        progress(f"Selecting listed call/put pairs: {i}/{len(tickers)} — {ticker}")
        try:
            if ticker not in current_spot:
                raise ValueError("No underlying price on the common as-of date.")
            try:
                chain = request_frame(bq, bq.univ.options(ticker), chain_items)
            except Exception:
                failed_chain_requests += 1
                raise
            completed_chain_requests += 1
            selected = select_pair(chain, float(current_spot[ticker]), asof, target_dte, tolerance)
            pairs.append({"Ticker": ticker, **selected})
            meta.loc[ticker, "Selection status"] = "OK"
        except Exception as exc:
            message = str(exc)
            meta.loc[ticker, "Selection status"] = message
            errors.append({"Stage": "Option selection", "Security": ticker, "Error": message})
            # Avoid hundreds of identical queries after an API/schema failure.
            if completed_chain_requests == 0 and failed_chain_requests >= 3:
                raise RuntimeError("The first option-chain requests failed. " + message) from exc
    if pairs:
        selected = pd.DataFrame(pairs).set_index("Ticker")
        meta = meta.join(selected)
    else:
        for col in ["Expiry", "Strike", "DTE", "Call contract", "Put contract"]:
            meta[col] = np.nan

    option_ids = sorted(set(meta["Call contract"].dropna()) | set(meta["Put contract"].dropna()))
    history_dates = bq.func.range("-2Y", asof.strftime("%Y-%m-%d"))
    # Concrete BQL data items: these are listed-option bid/ask prices, not IV.
    quote_items = {"Bid": bq.data.px_bid(dates=history_dates, fill="NA"),
                   "Ask": bq.data.px_ask(dates=history_dates, fill="NA")}
    quote_parts = []
    failed_option_ids = set()
    for i in range(0, len(option_ids), BATCH_SIZE):
        batch = option_ids[i:i+BATCH_SIZE]
        progress(f"Option bid/ask history: {min(i+BATCH_SIZE, len(option_ids))}/{len(option_ids)} contracts")
        try:
            quote_parts.append(request_frame(bq, batch, quote_items, history=True))
        except Exception as exc:
            if i == 0:
                raise RuntimeError(
                    "First historical PX_BID/PX_ASK request failed; check these "
                    "items and their history entitlement in BQNT. " + str(exc)
                ) from exc
            failed_option_ids.update(batch)
            errors.append({"Stage": "Option history", "Security": ", ".join(batch), "Error": str(exc)})
    quotes = (quote_midpoints(pd.concat(quote_parts, ignore_index=True))
              if quote_parts else pd.DataFrame(columns=["ID", "DATE", "Bid", "Ask", "Mid", "Zero bid"]))
    by_id = {key: group for key, group in quotes.groupby("ID")}
    by_stock = {key: group for key, group in spots.groupby("ID")}
    rows = []
    for ticker in tickers:
        spot = by_stock.get(ticker, pd.DataFrame(columns=["DATE", "spot"]))
        panel = spot[["DATE", "spot"]].set_index("DATE")
        for side, col in [("Call", "call_mid"), ("Put", "put_mid")]:
            option_id = meta.loc[ticker, f"{side} contract"]
            q = by_id.get(option_id, pd.DataFrame(columns=["DATE", "Mid", "Zero bid"]))
            series = q.set_index("DATE")["Mid"].rename(col)
            panel = panel.join(series, how="outer", validate="one_to_one")
            valid_dates = q.loc[q["Mid"].notna(), "DATE"]
            meta.loc[ticker, f"{side} last date"] = valid_dates.max() if not valid_dates.empty else pd.NaT
            on_date = q.loc[q["DATE"].eq(asof)]
            meta.loc[ticker, f"{side} zero bid"] = bool(on_date["Zero bid"].any())
            if option_id in failed_option_ids:
                meta.loc[ticker, "Selection status"] = "Option-history request failed; see query errors"
        panel = panel.reindex(panel.index.union(pd.DatetimeIndex([asof])))
        panel.index.name = "date"
        panel["ticker"] = ticker
        for col in ["spot", "call_mid", "put_mid"]:
            panel[col] = pd.to_numeric(panel[col], errors="raise").astype(float)
        rows.append(panel.reset_index())
    raw = validate_history(pd.concat(rows, ignore_index=True))
    return {"history": raw, "contracts": meta, "errors": pd.DataFrame(errors),
            "asof": asof, "target_dte": target_dte, "tolerance": tolerance}

## 3.5 四种表不要混淆：这是读代码最有用的地图

**依据：原单元格 7 和原单元格 5。**

| 阶段 / 对象 | 每行大致代表什么 | 关键列或索引 |
|---|---|---|
| `members` / `meta` | 一只当前成分股 | 股票 ID、名称、合约与状态 |
| `chain` | 某只股票的一张当前挂牌期权 | 合约 ID、Expiry、Strike、Side |
| `quotes` | 一张选中期权在某个日期的报价 | 合约 ID、DATE、Bid、Ask、Mid、Zero bid |
| `history` | 一只股票在某个日期的股价与所选期权价格 | date、ticker、spot、call_mid、put_mid |
| `price["call_mid"]` 等 | 一行是日期，每列是一只股票 | 日期 × 股票宽表 |
| `latest` | 一只股票在共同 asof 的摘要 | 原始价格、三种 Z、最近任一报价日期 |

**两类检查承担不同责任：** `request_frame` 保证证券/日期结果不因错误匹配而重复；`validate_history` 保证进入统计的 `date + ticker` 唯一。它们都不能代替对真实证券口径、权限和交割内容的外部核实。

### 取数失败并非都按同一方式处理

成员查询或股价历史查询没有局部容错分支，失败会使本次取数停止。单只股票选合约失败通常会记录并继续；期权链累计失败达到前述启动条件时停止。首批期权历史失败会停止，后续批次失败会记录并继续。最终界面的刷新函数会捕捉传播出来的错误并展示。

<a id="widgets"></a>
# 4. BQNT dashboard widgets：按钮、表格和图怎样连起来？

**对应原单元格 8、9。** `launch_dashboard(bq)` 建立交互界面，返回一个包含所有控件的 `VBox` 对象，不是返回行情 DataFrame。

初学者最需要理解的是：**创建按钮 ≠ 点击按钮；定义函数 ≠ 调用函数；重新画表格 ≠ 再次请求 Bloomberg。**

这个大函数内部还定义了四个小函数：`model`、`draw_table`、`draw_chart`、`refresh_data`。它们可以访问外层创建的控件和 `state`。这种组织让同一个界面的按钮、图表和缓存共享一份状态，而不用把所有中间变量放成全局变量。

<details>
<summary>展开查看原文件第 8 个单元格的英文说明（原文保留）</summary>

## 4. BQNT dashboard widgets

</details>

### 4.1 导入界面工具，建立控件和状态

**原文件定位：第 9 个单元格，第 1–25 行。以下是阅读片段，不要单独运行。**

```python
def launch_dashboard(bq=None):
    """Create notebook widgets. The Load button is the only data-loading trigger."""
    import bql
    import ipywidgets as w
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display
    bq = bq if bq is not None else bql.Service()
    target = w.IntText(value=TARGET_DTE, description="Target DTE:")
    band = w.IntText(value=EXPIRY_TOLERANCE_DAYS, description="DTE band ±:")
    basis = w.Dropdown(options=["Raw premium", "Premium / spot"], description="Basis:")
    metric = w.Dropdown(options=["Call Z", "Put Z", "Spread Z"], value="Call Z", description="Rank by:")
    minimum = w.FloatSlider(value=0, min=0, max=4, step=0.5, description="Min |Z|:", continuous_update=False)
    top_n = w.Dropdown(options=[20, 50, 100, 600], value=50, description="Max rows:")
    search = w.Text(placeholder="Ticker contains...", description="Search:", continuous_update=False)
    stock = w.Dropdown(options=[], description="Chart:", layout=w.Layout(width="330px"))
    refresh = w.Button(description="Load / refresh Bloomberg", icon="refresh", layout=w.Layout(width="240px"))
    progress = w.HTML()
    banner = w.HTML(value=(
        "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
        "<p>Click <b>Load / refresh Bloomberg</b>. No demo or CSV data path.<br>"
        "Each selected contract is compared with its own history. This is NOT a "
        "constant-maturity or historically rolled ATM series.</p>"
    ))
    table_out, chart_out, error_out = w.Output(), w.Output(), w.Output()
    state = {"data": None, "models": {}, "loading": False}
```

`ipywidgets as w` 提供下拉菜单、文本输入框、滑条和按钮；`matplotlib.pyplot as plt` 负责画图；`HTML` 与 `display` 负责在 Notebook 中展示内容。

`bq = bq if bq is not None else bql.Service()` 表示：调用者已经提供会话就使用它，否则创建会话。即使传入 `bq`，本函数仍包含 `import bql`，所以真实创建此界面需要对应环境。

`value` 是控件当前值，`description` 是标签，`options` 是可选项，`layout` 管位置与尺寸。`continuous_update=False` 表示不必在每次输入或拖动的中间步骤都触发更新。

`table_out`、`chart_out`、`error_out` 是三个独立输出区域，分别放表格、图形和错误。

`state` 里：`data=None` 表示还没拿到本次行情；`models={}` 用来按价格口径缓存计算结果；`loading=False` 表示目前没有加载任务。它不是保存到硬盘的数据库。

## 4.2 每个控件该怎么理解？

**依据：原单元格 9 的控件创建与事件绑定。**

| 界面文字 | 影响什么 | 是否发起新行情请求 |
|---|---|---|
| `Target DTE` | 下一次选合约的目标剩余天数 | 修改本身不请求；点击加载按钮后生效 |
| `DTE band ±` | 下一次选合约允许的到期距离 | 同上 |
| `Load / refresh Bloomberg` | 清除旧结果，重查成员、价格、合约与历史 | **是** |
| `Basis` | 原始价格或价格/股价百分比口径 | 否；首次使用该口径时本地计算并缓存 |
| `Rank by` | 按 Call Z、Put Z 或 Spread Z 排名 | 否；主要重画表格 |
| <code>Min &#124;Z&#124;</code> | 只保留绝对 Z 达到阈值的行 | 否 |
| `Max rows` | 排名表最多显示 20、50、100 或 600 行 | 否；不等于减少查询股票数 |
| `Search` | 股票代码包含输入文本，忽略大小写 | 否；不是搜索公司名称 |
| `Chart` | 选择要画图的股票 | 否；不保证与表格筛选结果同步 |

`Rank by` 即使选了 `Spread Z`，上方历史 Z 图仍只画 Call Z 与 Put Z；原实现没有画第三条 Spread Z 线。

### 4.3 `model()`：为什么调筛选条件不用反复算所有数据？

**原文件定位：第 9 个单元格，第 27–33 行。以下是阅读片段，不要单独运行。**

```python
    def model():
        if basis.value not in state["models"]:
            state["models"][basis.value] = compute_dashboard(
                state["data"]["history"], basis.value, WINDOW_DAYS, MIN_OBSERVATIONS,
                asof=state["data"]["asof"],
            )
        return state["models"][basis.value]
```

`basis.value` 是当前选择的价格口径。若 `state["models"]` 里没有该口径，才调用 `compute_dashboard`，使用已加载的历史、窗口、最少观察值和共同日期。

算出的整个结果包以口径名称为键存起来。下次只改搜索或排名等控件时，`model()` 可以直接返回已有结果；切换到另一种口径时最多再算一份。

这叫缓存：保存已经算过的结果来复用。它不缓存新行情，也不意味着行情自动刷新。按加载按钮时，原代码会清空这些缓存。

### 4.4 `draw_table()`：拼摘要、显示状态、筛选并排序

**原文件定位：第 9 个单元格，第 35–50 行。以下是阅读片段，不要单独运行。**

```python
    def draw_table(*_):
        if state["loading"] or state["data"] is None:
            return
        m = model()
        key = METRICS[metric.value]
        all_rows = m["latest"].join(state["data"]["contracts"])
        all_rows["Prior observations"] = m["counts"][key].loc[m["asof"]].astype(int)
        all_rows["Z status"] = m["statuses"][key]
        bad_selection = all_rows["Selection status"].ne("OK")
        all_rows.loc[bad_selection, "Z status"] = all_rows.loc[bad_selection, "Selection status"]
        t = all_rows.copy()
        if search.value.strip():
            t = t.loc[t.index.str.contains(search.value.strip(), case=False, regex=False)]
        if minimum.value > 0:
            t = t.loc[t[metric.value].abs().ge(minimum.value)]
        t = t.assign(_rank=t[metric.value].abs()).sort_values("_rank", ascending=False, na_position="last").head(top_n.value)
```

`*_` 表示接收控件回调传来的任意位置参数，但不使用它们。开头的 `return` 是保护：正在加载或没有数据时，不继续画表。

`all_rows` 把统计摘要与合约信息按股票索引合并。当前 `Rank by` 决定显示哪一种指标的先前观察值数量和状态。如果选合约/取历史阶段的 `Selection status` 不是 `OK`，这条取数原因会覆盖统计状态，优先告诉你上游发生了什么。

搜索使用 `index.str.contains(..., case=False, regex=False)`：在股票代码中按普通文字匹配，忽略大小写，不把用户输入当正则表达式。

只有最小阈值大于 0 时，才执行 `abs(Z) >= 阈值` 过滤。缺失 Z 在这个比较中不会通过。阈值为 0 时，不执行这一步，缺失行仍可排在后面。

`assign(_rank=...)` 创建临时绝对值排名列；按它降序排列，缺失排最后，再取前 `top_n.value` 行。所以 `Z=-3` 会排在 `Z=+2` 前面，不是只找正数最大的一边。

### 4.5 `draw_table()`：格式化展示，不改变原始价格

**原文件定位：第 9 个单元格，第 51–63 行。以下是阅读片段，不要单独运行。**

```python
        columns = ["Name", "Spot", "Expiry", "DTE", "Strike", "Call mid", "Put mid",
                   "Call Z", "Put Z", "Spread Z", "Prior observations", "Z status",
                   "Call last date", "Put last date", "Call zero bid", "Put zero bid",
                   "Call contract", "Put contract"]
        t = t.reindex(columns=columns)
        for col in ["Expiry", "Call last date", "Put last date"]:
            t[col] = pd.to_datetime(t[col]).dt.strftime("%Y-%m-%d").fillna("—")
        with table_out:
            table_out.clear_output(wait=True)
            display(HTML(f"<p><b>{all_rows[metric.value].notna().sum()}/{len(all_rows)} stocks have a valid {escape(metric.value)}.</b> "
                         "Missing latest prices are not carried forward. Z-score ranking is by absolute value.</p>"))
            display(HTML('<div style="overflow-x:auto;max-height:520px;overflow-y:auto">'
                         + t.to_html(float_format=lambda v: f"{v:,.2f}", na_rep="—", escape=True) + "</div>"))
```

`columns` 指定最终显示列及其顺序。日期转成 `YYYY-MM-DD` 文字，缺失日期显示为破折号；数值通过 `to_html` 格式化到两位小数，缺失数值也显示为破折号。

这些格式化只用于展示副本，不会把统计计算的精度改成两位小数。比如显示 `0.00` 并不总意味着底层数值严格等于 0。

`with table_out:` 指定下面输出放进表格区域。`clear_output(wait=True)` 替换旧表，避免每次改筛选都往下面追加一张。外层 HTML 让较宽或较长的表可以滚动。

顶部 `有效数量 / 总数量` 统计的是**所有已加载行**，不是你搜索之后、截取前 N 行之后的数量。`escape=True` 与 `escape(...)` 用于让数据/错误文字按文本显示，不当成任意 HTML 标签执行。

### 4.6 `draw_chart()` 上半部分：画同合约的历史 Z

**原文件定位：第 9 个单元格，第 65–86 行。以下是阅读片段，不要单独运行。**

```python
    def draw_chart(*_):
        if state["loading"] or state["data"] is None or not stock.value:
            return
        m, ticker = model(), stock.value
        left = m["asof"] - pd.Timedelta(days=365)
        info = state["data"]["contracts"].loc[[ticker]]
        with chart_out:
            chart_out.clear_output(wait=True)
            display(HTML(info[["Call contract", "Put contract", "Selection status"]].to_html(escape=True)))
            fig, ax = plt.subplots(figsize=(11, 3.2))
            for label, key in [("Call Z", "call"), ("Put Z", "put")]:
                z = m["scores"][key].loc[left:, ticker]
                ax.plot(z.index, z.values, label=label)
            ax.axhline(2, linestyle="--", linewidth=0.8)
            ax.axhline(-2, linestyle="--", linewidth=0.8)
            ax.axhline(0, linestyle=":", linewidth=0.8)
            ax.set_title(f"{ticker} | Same-contract rolling 365-day Z | {basis.value}")
            ax.set_ylabel("Standard deviations")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
```

只有已加载、没在加载中、且选了股票时才作图。`left = asof - 365 天` 是图的左边界；这段代码写死了 365 天。

`contracts.loc[[ticker]]` 用双层方括号保留“一行表”的结构，方便显示 Call/Put 合约 ID 与状态。不要只看股票名，也要看这里的合约是否改变。

`fig` 是整张图的容器，`ax` 是画线的坐标区域。循环分别取 Call 与 Put 的 Z 时间序列：`.loc[left:, ticker]` 表示从左边界至现有数据末尾，选这一只股票列。

`axhline(2)`、`axhline(-2)`、`axhline(0)` 只是画参考水平线。代码没有据此算交易信号、置信区间或显著性概率。历史得分为空时，相应日期不会产生有效曲线点。

`legend()` 显示图例，`tight_layout()` 调整边距，`show()` 显示，`close(fig)` 释放这张图，避免每次切换股票都留下一堆未关闭图形。

### 4.7 `draw_chart()` 下半部分：画原始期权价格

**原文件定位：第 9 个单元格，第 87–96 行。以下是阅读片段，不要单独运行。**

```python
            fig, ax = plt.subplots(figsize=(11, 3.0))
            for label, key in [("Call mid", "call_mid"), ("Put mid", "put_mid")]:
                p = m["price"][key].loc[left:, ticker]
                ax.plot(p.index, p.values, label=label)
            ax.set_title(f"{ticker} | Currently selected contracts' daily premiums")
            ax.set_ylabel("USD per underlying share")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
```

第二张图取的是 `m["price"]`，而不是 `m["values"]`，所以它始终显示原始 Call / Put 中间价，纵轴写的是每标的股对应的美元金额。

切换 `Basis` 会改变上方 Z 的计算口径，但这张原始价格图不会变成期权价格/股价百分比。表格的 Call mid / Put mid 也保留原始价格。

**参数修改时的一个实现细节：** 原图的 365 天显示区间与标题文字是固定写入的。直接修改全局 `WINDOW_DAYS` 会影响统计计算，却不会自动让这里所有图形文字和显示区间同步改变。本教学版保留这一实现，没有偷偷改掉它。

### 4.8 点击加载按钮后，先清空旧结果，再取新数据

**原文件定位：第 9 个单元格，第 98–115 行。以下是阅读片段，不要单独运行。**

```python
    def refresh_data(*_):
        state["loading"] = True
        state["data"], state["models"] = None, {}
        refresh.disabled = target.disabled = band.disabled = True
        table_out.clear_output()
        chart_out.clear_output()
        error_out.clear_output()
        try:
            data = fetch_bloomberg_history(
                bq, progress=lambda text: setattr(progress, "value", escape(text)),
                target_dte=int(target.value), tolerance=int(band.value), max_stocks=MAX_STOCKS,
            )
            state["data"] = data
            tickers = sorted(data["history"]["ticker"].unique())
            old = stock.value
            stock.options = tickers
            stock.value = old if old in tickers else tickers[0]
            loaded = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
```

先把 `loading` 设为 True，将旧的 `data` 清为 None、旧的计算缓存清空，并禁用加载按钮、目标 DTE 和容差输入框。然后清空三个输出区域。

接着调用 `fetch_bloomberg_history`。`lambda text: setattr(progress, "value", escape(text))` 把取数函数传来的进度文字写到 HTML 控件的 `value` 属性里。这里界面变量 `progress` 是控件，取数函数里的同名参数是通知函数，属于不同作用域。

`target.value` 和 `band.value` 用的是用户此刻输入的值，不是永远使用初始默认值。`max_stocks=MAX_STOCKS` 则显式传入全局上限。

成功之后，把结果包保存到 `state["data"]`，更新可画图的股票列表。之前选的股票还在，就保留；否则取排序后的第一只。`loaded` 记录的是本次加载时的 UTC 时间，与行情的共同市场日期 `asof` 是两件事。

### 4.9 更新说明、展示问题，并确保恢复界面

**原文件定位：第 9 个单元格，第 116–143 行。以下是阅读片段，不要单独运行。**

```python
            banner.value = (
                "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
                f"<p><b>Bloomberg BQL data | Common as-of date: {data['asof']:%Y-%m-%d}</b><br>"
                f"Loaded {loaded}. Latest available daily observations; NOT a streaming feed.<br>"
                f"Current listed pairs: target {data['target_dte']} DTE ± {data['tolerance']} days, "
                "closest eligible expiry then closest strike to spot; identical expiry/strike for call and put.<br>"
                f"Baseline: previous {WINDOW_DAYS} calendar days, excluding the current date; "
                f"at least {MIN_OBSERVATIONS} valid observations and a full elapsed year.<br>"
                "<b>Same-contract history, NOT constant-maturity history.</b> Recently listed contracts can have no valid Z-score. "
                "Check quote timing, spreads, deliverables and corporate actions before relying on this prototype.</p>"
            )
            progress.value = f"Loaded {len(tickers)} current constituent rows; {len(data['errors'])} query/selection issues."
            if not data["errors"].empty:
                with error_out:
                    display(HTML(data["errors"].to_html(index=False, escape=True)))
        except Exception as exc:
            banner.value = "<h3>Bloomberg loading failed</h3><p>" + escape(str(exc)) + "</p>"
            progress.value = "No substitute data was loaded."
        finally:
            state["loading"] = False
            refresh.disabled = target.disabled = band.disabled = False
        if state["data"] is not None:
            try:
                draw_table()
                draw_chart()
            except Exception as exc:
                with error_out:
                    display(HTML("<b>Rendering/calculation error:</b> " + escape(str(exc))))
```

`banner.value` 生成顶部说明：数据日期、加载时间、选合约目标、历史窗口和原型限制。`errors` 非空时，把详细问题表显示在错误区。

若取数失败，`except` 把错误放到顶部，并明确写“没有加载替代数据”。由于旧数据已在开头清空，失败后不会继续把旧价格冒充为这次成功加载的新价格。

`finally` 表示无论成功还是异常都执行：结束加载状态，并重新启用按钮及参数输入。

只有确实拿到了 `state["data"]`，才进入后续 `draw_table()` 与 `draw_chart()`。绘图/计算阶段也单独捕捉错误。这样可以区别“数据加载失败”和“已经加载、但计算或展示失败”。

### 4.10 事件绑定：到底是谁调用了这些内部函数？

**原文件定位：第 9 个单元格，第 145–155 行。以下是阅读片段，不要单独运行。**

```python
    for control in [metric, minimum, top_n, search]:
        control.observe(draw_table, names="value")
    stock.observe(draw_chart, names="value")
    basis.observe(lambda *_: (draw_table(), draw_chart()), names="value")
    refresh.on_click(refresh_data)
    errors_panel = w.Accordion(children=[error_out])
    errors_panel.set_title(0, "Query / selection issues")
    errors_panel.selected_index = None
    controls = w.Box([basis, metric, minimum, top_n, search], layout=w.Layout(display="flex", flex_flow="row wrap"))
    return w.VBox([banner, w.HBox([target, band, refresh]), progress, controls,
                   table_out, stock, chart_out, errors_panel])
```

`control.observe(draw_table, names="value")` 的意思不是“现在画表”，而是登记规则：以后该控件的 value 改变时，调用 `draw_table`。股票选择改变时只画图；价格口径改变时依次画表和图。

`refresh.on_click(refresh_data)` 把按钮点击事件与加载函数绑定。**这正是原文件中真正触发行情加载的入口。**

`Accordion` 是可展开的问题面板，`selected_index=None` 让它开始时折叠。`HBox` 横向排列，`VBox` 纵向排列，`Box` 加上布局设置让控件需要时换行。

最后 `return w.VBox([...])` 把所有部件装成一个完整界面，交给外层 `display` 显示。仅返回界面时还没有点击按钮，所以没有自动执行取数。

教学补充出处：[ipywidgets 官方事件文档（S3）](https://ipywidgets.readthedocs.io/en/8.1.4/examples/Widget%20Events.html)。其中 `observe` 对应属性变化通知，`on_click` 对应按钮点击回调；具体哪些动作触发哪些函数，以这里保留的原代码为准。

### 运行完整函数：`launch_dashboard`

下一格完整保留原单元格 9 的 155 行。**现在执行只是定义，默认离线流程不会创建 Bloomberg 界面。** 不要把前面的嵌套函数切片当成独立代码格运行：它们需要外层创建的控件和状态。

In [20]:
def launch_dashboard(bq=None):
    """Create notebook widgets. The Load button is the only data-loading trigger."""
    import bql
    import ipywidgets as w
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display
    bq = bq if bq is not None else bql.Service()
    target = w.IntText(value=TARGET_DTE, description="Target DTE:")
    band = w.IntText(value=EXPIRY_TOLERANCE_DAYS, description="DTE band ±:")
    basis = w.Dropdown(options=["Raw premium", "Premium / spot"], description="Basis:")
    metric = w.Dropdown(options=["Call Z", "Put Z", "Spread Z"], value="Call Z", description="Rank by:")
    minimum = w.FloatSlider(value=0, min=0, max=4, step=0.5, description="Min |Z|:", continuous_update=False)
    top_n = w.Dropdown(options=[20, 50, 100, 600], value=50, description="Max rows:")
    search = w.Text(placeholder="Ticker contains...", description="Search:", continuous_update=False)
    stock = w.Dropdown(options=[], description="Chart:", layout=w.Layout(width="330px"))
    refresh = w.Button(description="Load / refresh Bloomberg", icon="refresh", layout=w.Layout(width="240px"))
    progress = w.HTML()
    banner = w.HTML(value=(
        "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
        "<p>Click <b>Load / refresh Bloomberg</b>. No demo or CSV data path.<br>"
        "Each selected contract is compared with its own history. This is NOT a "
        "constant-maturity or historically rolled ATM series.</p>"
    ))
    table_out, chart_out, error_out = w.Output(), w.Output(), w.Output()
    state = {"data": None, "models": {}, "loading": False}

    def model():
        if basis.value not in state["models"]:
            state["models"][basis.value] = compute_dashboard(
                state["data"]["history"], basis.value, WINDOW_DAYS, MIN_OBSERVATIONS,
                asof=state["data"]["asof"],
            )
        return state["models"][basis.value]

    def draw_table(*_):
        if state["loading"] or state["data"] is None:
            return
        m = model()
        key = METRICS[metric.value]
        all_rows = m["latest"].join(state["data"]["contracts"])
        all_rows["Prior observations"] = m["counts"][key].loc[m["asof"]].astype(int)
        all_rows["Z status"] = m["statuses"][key]
        bad_selection = all_rows["Selection status"].ne("OK")
        all_rows.loc[bad_selection, "Z status"] = all_rows.loc[bad_selection, "Selection status"]
        t = all_rows.copy()
        if search.value.strip():
            t = t.loc[t.index.str.contains(search.value.strip(), case=False, regex=False)]
        if minimum.value > 0:
            t = t.loc[t[metric.value].abs().ge(minimum.value)]
        t = t.assign(_rank=t[metric.value].abs()).sort_values("_rank", ascending=False, na_position="last").head(top_n.value)
        columns = ["Name", "Spot", "Expiry", "DTE", "Strike", "Call mid", "Put mid",
                   "Call Z", "Put Z", "Spread Z", "Prior observations", "Z status",
                   "Call last date", "Put last date", "Call zero bid", "Put zero bid",
                   "Call contract", "Put contract"]
        t = t.reindex(columns=columns)
        for col in ["Expiry", "Call last date", "Put last date"]:
            t[col] = pd.to_datetime(t[col]).dt.strftime("%Y-%m-%d").fillna("—")
        with table_out:
            table_out.clear_output(wait=True)
            display(HTML(f"<p><b>{all_rows[metric.value].notna().sum()}/{len(all_rows)} stocks have a valid {escape(metric.value)}.</b> "
                         "Missing latest prices are not carried forward. Z-score ranking is by absolute value.</p>"))
            display(HTML('<div style="overflow-x:auto;max-height:520px;overflow-y:auto">'
                         + t.to_html(float_format=lambda v: f"{v:,.2f}", na_rep="—", escape=True) + "</div>"))

    def draw_chart(*_):
        if state["loading"] or state["data"] is None or not stock.value:
            return
        m, ticker = model(), stock.value
        left = m["asof"] - pd.Timedelta(days=365)
        info = state["data"]["contracts"].loc[[ticker]]
        with chart_out:
            chart_out.clear_output(wait=True)
            display(HTML(info[["Call contract", "Put contract", "Selection status"]].to_html(escape=True)))
            fig, ax = plt.subplots(figsize=(11, 3.2))
            for label, key in [("Call Z", "call"), ("Put Z", "put")]:
                z = m["scores"][key].loc[left:, ticker]
                ax.plot(z.index, z.values, label=label)
            ax.axhline(2, linestyle="--", linewidth=0.8)
            ax.axhline(-2, linestyle="--", linewidth=0.8)
            ax.axhline(0, linestyle=":", linewidth=0.8)
            ax.set_title(f"{ticker} | Same-contract rolling 365-day Z | {basis.value}")
            ax.set_ylabel("Standard deviations")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(11, 3.0))
            for label, key in [("Call mid", "call_mid"), ("Put mid", "put_mid")]:
                p = m["price"][key].loc[left:, ticker]
                ax.plot(p.index, p.values, label=label)
            ax.set_title(f"{ticker} | Currently selected contracts' daily premiums")
            ax.set_ylabel("USD per underlying share")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)

    def refresh_data(*_):
        state["loading"] = True
        state["data"], state["models"] = None, {}
        refresh.disabled = target.disabled = band.disabled = True
        table_out.clear_output()
        chart_out.clear_output()
        error_out.clear_output()
        try:
            data = fetch_bloomberg_history(
                bq, progress=lambda text: setattr(progress, "value", escape(text)),
                target_dte=int(target.value), tolerance=int(band.value), max_stocks=MAX_STOCKS,
            )
            state["data"] = data
            tickers = sorted(data["history"]["ticker"].unique())
            old = stock.value
            stock.options = tickers
            stock.value = old if old in tickers else tickers[0]
            loaded = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
            banner.value = (
                "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
                f"<p><b>Bloomberg BQL data | Common as-of date: {data['asof']:%Y-%m-%d}</b><br>"
                f"Loaded {loaded}. Latest available daily observations; NOT a streaming feed.<br>"
                f"Current listed pairs: target {data['target_dte']} DTE ± {data['tolerance']} days, "
                "closest eligible expiry then closest strike to spot; identical expiry/strike for call and put.<br>"
                f"Baseline: previous {WINDOW_DAYS} calendar days, excluding the current date; "
                f"at least {MIN_OBSERVATIONS} valid observations and a full elapsed year.<br>"
                "<b>Same-contract history, NOT constant-maturity history.</b> Recently listed contracts can have no valid Z-score. "
                "Check quote timing, spreads, deliverables and corporate actions before relying on this prototype.</p>"
            )
            progress.value = f"Loaded {len(tickers)} current constituent rows; {len(data['errors'])} query/selection issues."
            if not data["errors"].empty:
                with error_out:
                    display(HTML(data["errors"].to_html(index=False, escape=True)))
        except Exception as exc:
            banner.value = "<h3>Bloomberg loading failed</h3><p>" + escape(str(exc)) + "</p>"
            progress.value = "No substitute data was loaded."
        finally:
            state["loading"] = False
            refresh.disabled = target.disabled = band.disabled = False
        if state["data"] is not None:
            try:
                draw_table()
                draw_chart()
            except Exception as exc:
                with error_out:
                    display(HTML("<b>Rendering/calculation error:</b> " + escape(str(exc))))

    for control in [metric, minimum, top_n, search]:
        control.observe(draw_table, names="value")
    stock.observe(draw_chart, names="value")
    basis.observe(lambda *_: (draw_table(), draw_chart()), names="value")
    refresh.on_click(refresh_data)
    errors_panel = w.Accordion(children=[error_out])
    errors_panel.set_title(0, "Query / selection issues")
    errors_panel.selected_index = None
    controls = w.Box([basis, metric, minimum, top_n, search], layout=w.Layout(display="flex", flex_flow="row wrap"))
    return w.VBox([banner, w.HBox([target, band, refresh]), progress, controls,
                   table_out, stock, chart_out, errors_panel])

## 4.11 实际运行顺序与文件阅读顺序不同

**依据：原单元格 7、9、11 的函数调用。**

```text
从上到下运行定义：准备工具、参数、7 个顶层函数
    ↓
启动格：bql.Service() → launch_dashboard(bq) → display(dashboard)
    ↓
此时只出现界面，还没有加载行情
    ↓
用户点击 Load / refresh Bloomberg
    ↓
refresh_data()
    ├─ fetch_bloomberg_history()
    │      ├─ request_frame() 多次：成员、股价、期权链、报价历史
    │      ├─ select_pair() 多次：逐股票挑选合约
    │      ├─ quote_midpoints()：过滤报价并计算 Mid
    │      └─ validate_history()：检查最终长表
    ├─ 保存 data 并清理/更新界面状态
    ├─ draw_table() → model() → 首次 compute_dashboard()
    │                                  └─ validate_history()
    └─ draw_chart() → model() → 复用相同口径的结果
```

以后只改 `Search`：主要重新画表。只改 `Chart`：主要重新画图。改 `Basis`：需要时计算另一种口径后更新表和图。再次点击加载按钮：才会重走取数流程，并可能选出新的合约。

<a id="launch"></a>
# 5. Launch：怎样运行这份教学版？

**对应原单元格 10、11。**

### 路径 A：暂时没有 Bloomberg，只学习代码

保持下方 `RUN_BLOOMBERG = False`，从上到下执行即可。你会看到教学练习输出和“已跳过 Bloomberg 启动”的说明，不会创建 Bloomberg 会话或请求行情。

这条路径依赖 Python、pandas、NumPy 与 Jupyter/IPython；本文件已保存本次离线练习输出。它不表示原仪表盘新增了模拟数据模式。

### 路径 B：在有权限的 Bloomberg BQNT 中运行真实界面

先在设置单元格将 `MAX_STOCKS` 设为 `5` 做初次连接检查，并按顺序重新执行设置与函数定义。再将下方开关改为 `True`、执行启动格。界面出现后，点击 **Load / refresh Bloomberg**。

先核对共同日期、选中合约、两个最近报价日期和错误面板，再决定是否把上限恢复为 `None`。这里复述的是原文件的连接检查流程，不是对真实请求成功的保证。

**不要因为 `import bql` 失败，就随便安装网上某个同名包并期待获得 Bloomberg 权限。** 原文件要求的是有相应数据与 API 权限的 BQuant 环境；普通 Notebook 中的通用 Python 库不能替代该连接条件。

<details>
<summary>展开查看原文件第 10 个单元格的英文说明（原文保留）</summary>

## 5. Launch

Run this cell, then click **Load / refresh Bloomberg**.

</details>

原文件第 11 个单元格只有下面 6 行。先逐行理解，再运行带开关的版本：

```python
import bql
from IPython.display import display

bq = bql.Service()
dashboard = launch_dashboard(bq)
display(dashboard)
```

| 语句 | 做什么 |
|---|---|
| `import bql` | 导入原程序使用的 Bloomberg 查询接口 |
| `from IPython.display import display` | 准备 Notebook 显示函数 |
| `bq = bql.Service()` | 创建 BQL 服务对象 |
| `dashboard = launch_dashboard(bq)` | 创建控件、内部函数、缓存和事件连接 |
| `display(dashboard)` | 将界面显示出来；仍要点击按钮才取行情 |

下面的开关是**教学版新增**，原启动语句保留在 `if` 分支内，只增加了缩进。运行失败时不会转换成教学行情，也不会吞掉真实连接异常。

In [21]:
# 教学版新增：默认只学习，不连接 Bloomberg。
RUN_BLOOMBERG = False

if RUN_BLOOMBERG:
    import bql
    from IPython.display import display

    bq = bql.Service()
    dashboard = launch_dashboard(bq)
    display(dashboard)
else:
    print("教学阅读：已跳过 Bloomberg 启动。前面的离线练习与原计算函数仍可运行。")

教学阅读：已跳过 Bloomberg 启动。前面的离线练习与原计算函数仍可运行。


## 5.1 改了参数却没变化，可能是哪里没重新执行？

**依据：原函数签名、缓存与事件绑定；语法说明是教学补充。**

Python 函数的默认参数通常在**执行函数定义时**确定。例如 `fetch_bloomberg_history` 的默认 `target_dte=TARGET_DTE`，不是一个会自动实时追踪全局变量的公式。不过原界面的刷新函数又显式传入当前控件值，因此用户在界面改 DTE 后点击刷新，就会使用新输入。

控件创建时也会读取初始全局配置。已经创建好的控件，不会因为后来你改了一格代码里的数字，就自动改变显示值。与此同时，计算结果按 `Basis` 缓存；仅改变全局窗口变量而不清缓存，也不保证马上重算。

最容易理解的做法是：修改配置后，重新运行相应设置、完整函数定义和启动格，再点击加载。不要在许多旧界面都存在时混用不同界面上的按钮。注意重启内核后需要从头重新执行定义。

补充参考：[Python 官方控制流教程（S4）](https://docs.python.org/3/tutorial/controlflow.html) 中函数定义、默认参数与 `lambda` 的说明。它仅辅助解释 Python 行为，不替原文件改变运行逻辑。

<a id="interpretation"></a>
# 6. Interpretation and data checks：怎样读结果才不会误会？

**对应原单元格 12，并结合实际计算和展示语句解释。** 这一节不增加新的交易判断，只说明原程序输出能表达什么、不能表达什么。

<details>
<summary>展开查看原文件第 12 个单元格的英文说明（原文保留）</summary>

## Interpretation and data checks

Prices are daily bid/ask midpoints, not implied volatilities and not theoretical model prices. The application rejects missing, negative or crossed quotes; zero bids with positive asks are retained and flagged. No missing observations are forward-filled. The displayed as-of date is the latest date returned in the underlying history; it is not proof that every exchange quote was refreshed at that instant.

Call and put selection requires identical expiry and strike and rejects duplicate expiry/strike/side rows. This does **not** by itself verify identical deliverables or standard contract adjustments. Review corporate actions, adjusted contracts, quote timestamps, liquidity and the chosen pricing source before relying on the output. Refresh can select a different current contract; the historical chart then refers to that newly selected contract throughout.

The ranking is by absolute Z-score. A large score means an unusual premium relative to the specified historical baseline, not a buy/sell recommendation. Spread Z is the Z-score of call-minus-put premiums, not the difference between the two separate Z-scores.

Query and selection failures are exposed in the expandable error panel. Unsupported history requests do not fall back to invented data or implied volatility.

### Reference material

- Bloomberg BQuant platform: https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/
- pandas date-based rolling windows: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html
- Published first-hand BQL option-chain example, including `options`, `expire_dt`, `strike_px` and `put_call`: https://medium.com/@annievelcu/visualising-the-effects-of-the-russo-ukrainian-war-in-the-markets-case-study-with-bql-and-bquant-bc908e782206

These public sources do not verify your live BQL version or historical PX_BID/PX_ASK entitlement. Use the field documentation in your Bloomberg session to resolve any rejected query.


</details>

## 6.1 结果表每一列怎么读？

| 界面列 | 含义 | 需要特别留意的地方 |
|---|---|---|
| `Name` / 行索引 | 名称 / 股票代码 | 以这次返回的当前成分股为范围 |
| `Spot` | 共同 `asof` 的股价 | 不是期权价格 |
| `Expiry` / `DTE` / `Strike` | 当前选中的到期日 / 剩余日历天 / 行权价 | 改目标并刷新后可能变 |
| `Call mid` / `Put mid` | 共同日期的原始中间价 | 当日缺失不会改显示最后一次旧报价 |
| `Call Z` / `Put Z` | 两张合约各自指标的历史标准化结果 | 跟随 `Basis` 口径，历史门槛分别检查 |
| `Spread Z` | Call−Put 差值序列的 Z | 不是两个 Z 相减，也不是 Bid−Ask |
| `Prior observations` | 当前排名指标的窗口内先前有效数量 | 排名指标变了，这列可能也变 |
| `Z status` | 当前排名指标的状态，或上游选择/取数错误 | 一条状态可能覆盖其他同时存在的问题 |
| `Call last date` / `Put last date` | 两边各自最近一次有效 Mid 的日期 | 两个日期可能不同，也可能早于 asof |
| `Call zero bid` / `Put zero bid` | asof 当天是否存在合格的零 Bid 报价 | False 不等于保证有一个正数 Bid |
| `Call contract` / `Put contract` | 实际用于历史请求的两个合约 ID | 比较两次刷新前，应先核对这两列 |

中间价是 Bid 与 Ask 的计算结果，不是模型理论价格，也不保证某笔订单可以按此成交。原检查没有验证报价时效、流动性、实际交割内容和所有公司行动影响。

## 6.2 “为什么有些股票没有 Z-score？”

**依据：原单元格 5 的状态覆盖、原单元格 7 的选择/取数状态、原单元格 9 的展示覆盖。**

| 界面可能显示的文字 | 中文解释 | 对应逻辑 |
|---|---|---|
| `OK` | 当前指标通过这套程序的展示条件 | 不代表交易建议或完整数据审计通过 |
| `No value at the as-of date` | 共同日期这个指标没有值 | 当天任一必要价格缺失都会影响对应指标 |
| `Less than 365 days of history` | 从输入历史中该指标第一个有效值到当前还不足 365 天 | 看时间跨度，不只是行数 |
| `Fewer than 200 prior observations` | 前 365 天里非缺失的先前值不足 200 个 | 当天值不计入 200 个 |
| `Zero/invalid standard deviation` | 历史标准差缺失、为 0 或未大于阈值 | 不会将无法计算的 Z 写成 0 |
| 选不到配对的说明 | 没有满足期限、行权价与方向要求的唯一配对 | 当前链与筛选规则的问题 |
| `Option-history request failed; see query errors` | 某批历史报价请求失败 | 展开问题面板看具体原因 |

**空白不是 0，也不一定是代码坏了。** 原作者选择“条件不够就不给分”，没有用短历史伪装一整年的比较。

即使 `Prior observations` 已达到 200，仍可能因不足完整 365 天、当前值缺失或标准差无效而没有分数。反过来，有超过一年股价历史，也不能证明选中的那张期权有一年有效报价。

## 6.3 用原来的 365 天 / 200 次规则再验证一次〔教学练习〕

下面构造三个**完全虚构、按日历日逐日排列、并非交易日历**的样本。它们都用原默认 `365 / 200` 规则，没有降低门槛。

第一组有 400 天的有效指标；第二组只有 210 天，观察值数量可能已超过 200，但时间跨度不足；第三组跨度也有 400 天，却只有很少的近期有效值。我们只观察 Call 这一列，不把输出当作行情分析。

In [22]:
# 教学练习 12：区分“窗口内的观察值数量”与“从首个有效值起的时间跨度”。
lesson_gate_full = pd.DataFrame({
    "date": pd.date_range("2023-01-01", periods=400, freq="D"),
    "ticker": ["GATE_DEMO"] * 400,
    "spot": [100.0] * 400,
    "call_mid": 10.0 + np.sin(np.arange(400) / 8.0),
    "put_mid": 8.0 + np.cos(np.arange(400) / 8.0),
})
lesson_gate_short = lesson_gate_full.tail(210).copy()
lesson_gate_sparse = lesson_gate_full.copy()
lesson_gate_sparse.loc[1:299, "call_mid"] = np.nan
lesson_gate_rows = []
for lesson_case, lesson_input in [
    ("400天、足够有效值", lesson_gate_full),
    ("仅210天、观察值已超200", lesson_gate_short),
    ("400天跨度、近期有效值太少", lesson_gate_sparse),
]:
    lesson_result = compute_dashboard(lesson_input)  # 不传短窗口：使用原默认365/200
    lesson_gate_rows.append({
        "教学情况": lesson_case,
        "先前有效数量": int(lesson_result["counts"]["call"].iloc[-1, 0]),
        "Call_Z是否有效": bool(pd.notna(lesson_result["latest"].iloc[0]["Call Z"])),
        "状态": lesson_result["statuses"]["call"].iloc[0],
    })
lesson_gates = pd.DataFrame(lesson_gate_rows)
display(lesson_gates)
assert lesson_gates["先前有效数量"].tolist() == [365, 209, 99]
assert lesson_gates["Call_Z是否有效"].tolist() == [True, False, False]

,教学情况,先前有效数量,Call_Z是否有效,状态
0,400天、足够有效值,365,True,OK
1,仅210天、观察值已超200,209,False,Less than 365 days of history
2,400天跨度、近期有效值太少,99,False,Fewer than 200 prior observations


## 6.4 选 `Premium / spot` 后到底哪部分变化了？

**依据：原单元格 5 的 `values`、`price` 和 `latest`，以及原单元格 9 的两张图。**

用于计算的指标改成 `100 × 期权中间价 / 当日股价`。它只调整价格尺度，没有把历史合约变成每天都距到期 30 天，也没有让行权价自动跟着股价滚动。

如果股价比例随时间变化，归一化序列的 Z 也可能变化。若某条序列只是全程乘以同一个正数，均值和标准差会按同一倍数变化，其 Z 不会仅因为这个固定缩放而改变；所以不能笼统说“选百分比一定让所有 Z 都不同”。

下一格沿用前面的六行教学数据，仅改变某些日期的虚构股价。最后一天 `Call mid=6`、`spot=200`，因此用于计算的百分比值是 3，但原始价格列仍为 6。

In [23]:
# 教学练习 13：百分比口径变化，但原始价格列保留。
lesson_basis_history = lesson_history.copy()
lesson_basis_history["spot"] = [100., 100., 200., 100., 100., 200.]
lesson_normalized = compute_dashboard(
    lesson_basis_history, basis="Premium / spot", window_days=5, min_obs=3
)
print("用于计算的当日Call百分比值:", lesson_normalized["values"]["call"].iloc[-1, 0])
print("latest里展示的原始Call mid:", lesson_normalized["latest"].iloc[0]["Call mid"])
print("百分比口径的Call Z:", round(lesson_normalized["latest"].iloc[0]["Call Z"], 6))
assert lesson_normalized["values"]["call"].iloc[-1, 0] == 3.0
assert lesson_normalized["latest"].iloc[0]["Call mid"] == 6.0
assert lesson_normalized["price"]["call_mid"].iloc[-1, 0] == 6.0

用于计算的当日Call百分比值: 3.0
latest里展示的原始Call mid: 6.0
百分比口径的Call Z: 0.174667


## 6.5 初学者常见报错与应检查的地方

这张表对应原代码的依赖和异常处理，不承诺能覆盖你的 Bloomberg 环境中的所有情况。

| 现象 | 先理解原因 | 应检查什么 |
|---|---|---|
| `NameError`，例如 `pd` 或函数名未定义 | 依赖的单元格尚未执行，或内核已重启 | 从设置与完整函数定义开始依次运行 |
| `No module named bql` | 当前环境没有原程序所需的接口 | 在普通环境保持教学开关关闭；真实连接需对应 BQNT 环境 |
| DataFrame 为空或缺列的 `ValueError` | 输入不满足五列结构 | 列名、大小写、空表，以及传入的是不是正确对象 |
| 日期时区 / 重复日期股票组合报错 | 不是原要求的日度市场日期结构 | 在数据源处确认日期语义；不要直接删除错误来掩盖口径问题 |
| 开启真实启动后，界面不显示或只显示对象文字 | 可能涉及 Notebook 前端对控件的支持 | 检查环境与前端；本教学版未验证该现场行为 |
| 点击加载后出现 `Bloomberg loading failed` | 取数异常已由刷新函数显示 | 阅读完整错误文字与对应字段/权限文档 |
| 表格存在，但很多行是破折号 | 可能是当日价格、历史长度、有效数量或标准差不合格 | `Z status`、`Prior observations`、两边最近报价日期 |
| 显示有效数量大于当前表格行数 | 顶部统计全体加载股票，表格还经过筛选与截断 | `Search`、<code>Min &#124;Z&#124;</code>、`Max rows` |
| 改目标天数后价格没立刻变 | 改控件不是加载动作 | 点击加载按钮，重新选择合约并读取历史 |

对真实证券的币种、合约调整、流动性和报价时效，原文件没有完成自动验证。这里应保留“不知道/未验证”的界限，不能把程序没报错理解为这些方面都没有问题。

<a id="review"></a>
# 7. 自测与速查：把这份程序讲给别人听

## 7.1 十个小问题

先自己回答，再展开答案。问题只围绕本文件，不需要额外学习定价模型。

1. 运行一个 `def` 单元格，会立即执行函数里的 Bloomberg 查询吗？
2. `INDEX = "SPX Index"` 是否表示后面只查询 SPX 指数本身的期权？
3. 默认目标为 30、容差为 14、最短 DTE 为 7，允许的默认 DTE 区间是什么？
4. 一只股票有 210 个有效先前值，但首个有效值仅在 250 天前，是否满足原默认历史条件？
5. `closed="left"` 是否让今天的价格参与今天的历史均值？
6. `Spread Z` 能否直接用 `Call Z - Put Z` 算出来？
7. Bid 为 0、Ask 为 0.4 时，程序会做什么？Bid 为 3、Ask 为 2 呢？
8. Call 昨天有价格、今天没有，界面是否把昨天价格拿来作为今天价格？
9. 在同一到期日，98 与 102 距现价 100 同样近，且都有合法配对，优先选哪一个？
10. 改 `Rank by` 为 Spread Z 后，历史 Z 图会自动增加 Spread Z 曲线吗？

<details>
<summary>展开答案</summary>

1. 不会。定义只是创建函数；函数被调用时才执行其函数体。
2. 不是。此标识用于获取当前成分股，随后按每只成分股查询期权链。
3. 16–44 个日历日，含两端；还必须有有效 Call / Put 配对。
4. 不满足完整 365 天跨度要求；观察值数量足够也不能替代这道门槛。
5. 不参与，窗口是 `[t-365天, t)`。
6. 不能。必须先得到 Call−Put 序列，再用该序列自己的历史均值与标准差标准化。
7. 前者 Mid 为 0.2 且零买价标记为 True；后者是交叉报价，Mid 留空。
8. 不会向前填充。最后有效日期可另行显示，但当前值仍为空。
9. 在前面的到期日优先条件都相同后，98 排在前面。
10. 不会。原上图仍只有 Call Z、Put Z 两条指标线。

</details>

## 7.2 看到这些 Python / pandas 写法，怎么翻译成中文？

**这是阅读原代码的语法速查，不是新算法。**

| 写法 | 读法 | 本文件中的例子 |
|---|---|---|
| `x = ...` / `x == ...` | 赋值 / 判断相等 | 创建参数 / 检查值 |
| `obj.attr` / `obj.method(...)` | 访问属性 / 调用方法 | `raw.empty` / `d.copy()` |
| `d["col"]` / `d[["a", "b"]]` | 一列 Series / 多列 DataFrame | 取日期 / 取两边价格 |
| `d.loc[条件]` | 按条件筛选行 | 留下 date ≤ asof |
| `d.loc[日期, 股票]` | 按标签找位置 | 取某日某只股票得分 |
| `d.iloc[0]` | 按位置取第一行 | 取排序后的最佳配对 |
| `.eq/.ne/.gt/.ge/.lt/.le` | 等于/不等于/大于/大于等于/小于/小于等于 | 价格与期限过滤 |
| `&` / <code>&#124;</code> / `~` | 逐项并且 / 或者 / 取反 | 合并布尔条件、排除重复 |
| `.isna()` / `.notna()` | 是否缺失 / 是否非缺失 | 日期检查、有效报价 |
| `.any()` | 是否至少一个成立 | 检查是否存在问题 |
| `.any(axis=1)` | 按行检查是否有任一列成立 | 两边期权任一有值 |
| `.where(条件)` | 条件成立保留，否则默认缺失 | 不合格报价或 Z 留空 |
| `.groupby(...)` | 按键分组 | 按证券分历史、取最近报价日期 |
| `.join/.merge` | 按索引或键把字段匹配到一起 | 连接报价与股价 |
| `pd.concat(...)` | 将一批同类表连接起来 | 汇总多个请求批次 |
| `f"...{x}..."` | 把变量值嵌入文字 | 进度、错误、图标题 |
| `lambda ...: ...` | 小型匿名函数 | 合并规则、事件回调 |
| `try / except / finally` | 尝试 / 出错处理 / 无论成败都收尾 | 加载行情并恢复控件 |
| `return` / `raise` | 返回结果 / 抛出异常 | 成功交付表格 / 拒绝坏数据 |

注意：表格里的 `|` 在不同上下文可用于集合并集或逐项逻辑“或”；不要把它与 Python 普通标量逻辑的 `or` 混为一谈。原代码使用 `set(...) | {...}` 时是在合并集合。

## 7.3 四个“停下来想一想”的练习

**练习 A：** 只把上面的虚构 `lesson_history` 最后一天 Call 改为缺失，先预测 Call Z、Put Z、Spread Z 哪些受影响，再运行计算函数核对。它不应该改变原始教学表，使用 `.copy()`。

**练习 B：** 把教学报价中的 Ask 从 2.4 改到 1.8、Bid 保持 2.0。先判断会不会生成一个 1.9 的有效 Mid，再运行 `quote_midpoints` 检查。

**练习 C：** 在虚构链里加入一对距目标同样近、但到期日更早的合约。先按完整四层优先级选，再运行 `select_pair`。不要只看行权价距离。

**练习 D：** 不执行任何代码，用一句话说明为什么“已经请求两年历史”不代表“每只股票都有一年有效期权 Z-score”。

<details>
<summary>核对思路</summary>

A：Call 与 Spread 当前值会缺失，相关 Z 留空；Put 是否有效仍独立按它自身检查。B：交叉报价会使 Mid 缺失，不会接受算术上的 1.9。C：到期距离先比较，再比较实际到期日，然后才是行权价距离与行权价。D：请求区间只指定想要的范围；当前合约可能上市较晚、报价缺失或标准差无效，并且必须在 asof 有当前值。

</details>

### 现在可以用这段话概括原文件

> 它从 Bloomberg 获取当前 S&P 500 成分股，为每只股票选择当前一对同到期日、同行权价的近似平值期权。之后查询这两张固定合约的历史买卖报价，过滤不合格报价并算中间价；对每个指标使用不含当日的过去 365 日历天基准，在观察值数量、历史跨度和标准差满足条件时算 Z-score。最后用交互表格和图展示结果。它不是恒定 30 天期权序列、不是回测，也不会自动交易。

<a id="sources"></a>
# 8. 来源、原文件映射与本次验证

## 8.1 哪些是原文件内容，哪些是教学补充？

**主要依据：** 用户上传的 `BQNT_SP500_Options_ZScore(1).ipynb`。本文件各段“原文件定位”都以原文件的单元格序号和该单元格内从 1 开始的源代码行号为准，不是原文件 JSON 的物理行号。

**原文保留：** 原 7 个 Markdown 单元格置于相应可展开区域；原设置与 7 个顶层函数以可执行代码保留。为了让完整函数便于逐一运行，原长代码格按函数边界拆开。原最后 6 行启动语句原样展示，并保留在默认关闭的 `if RUN_BLOOMBERG` 分支中。

**教学补充：** 中文语法解释、类比、数据结构小表、13 个虚构数据练习、自测问答、启动开关及其提示信息。这些内容用来说明原代码，不表示原文件原本已有这些示例，也不表示真实证券存在示例中的报价。

**从实现推得的提醒：** 例如实际排序的平局规则、asof 不是所有股票都有数据的交集、状态覆盖优先级、百分比口径下原价格仍保留、图中不画 Spread Z 等，均依据保留的具体语句推导。未将这些提醒改写成原程序没有实施的“修复”。

| 原文件单元格 | 原内容 | 本教学版位置 |
|---|---|---|
| 1 | 方法、边界和原作者验证说明 | 封面、0、6 |
| 2–3 | 设置说明、导入、常量 | 1 |
| 4–5 | 输入检查、滚动 Z 计算 | 2 |
| 6–7 | BQL 合并、选合约、报价过滤、历史组装 | 3 |
| 8–9 | 控件、计算缓存、表格、图形、刷新和事件绑定 | 4 |
| 10–11 | 启动说明和 6 行启动代码 | 5 |
| 12 | 解读限制和原参考资料 | 6，并保留原文 |

## 8.2 仅用于解释通用语法的官方补充资料

S1：[pandas：DataFrame.rolling](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) — 核对日期窗口、`closed` 和 `min_periods` 的含义。

S2：[pandas：Rolling.std](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Rolling.std.html) — 核对 `ddof=1` 对应 `n-1` 分母。

S3：[ipywidgets：Widget Events](https://ipywidgets.readthedocs.io/en/8.1.4/examples/Widget%20Events.html) — 核对属性变化与按钮点击的回调方式。

S4：[Python：More Control Flow Tools](https://docs.python.org/3/tutorial/controlflow.html) — 辅助阅读函数定义、默认参数与 `lambda`。

这些资料只补充通用工具的含义，**不证明你的 Bloomberg 接口版本、历史 Bid / Ask 覆盖、填充语义或许可证已通过验证**。原文保留的 Bloomberg / BQL 参考地址也不构成这种保证。

## 8.3 本次执行与原代码保留检查

本教学版采用默认关闭的 Bloomberg 启动开关执行。下方记录的是本次本地教学执行环境；它不是用户的 Bloomberg 环境版本。

代码片段与原文件做了逐段文本核对；将原设置和拆开的函数合并后，其 Python 语法树与原对应代码一致；原启动分支里的语句与原 6 行启动代码一致。新增的 `assert` 是离线练习的检查条件，任何一个不成立都会使该练习报错，而不是伪造“通过”输出。

**未做的验证：** 没有执行 `bq.execute`，没有调用真实 `bql.Service()`，没有加载真实行情，没有验证 BQNT 的控件渲染，也没有验证数据权限、证券交割属性或服务端历史填充行为。

In [24]:
# 教学执行记录：只报告当前 Python / 通用计算库，不查询行情。
import sys as lesson_sys
print("Python:", lesson_sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Bloomberg 启动开关:", RUN_BLOOMBERG)
print("离线教学执行到此完成；Bloomberg 连接和真实行情未在本次生成中验证。")

Python: 3.13.5
pandas: 2.2.3
NumPy: 2.3.5
Bloomberg 启动开关: False
离线教学执行到此完成；Bloomberg 连接和真实行情未在本次生成中验证。
